In [2]:
# --- Data Loading and Preparation ---
from google.colab import drive
drive.mount('/content/drive')
import numpy as np
import pandas as pd

Mounted at /content/drive


# Data Loading and Preparation

In [3]:
# --- Data Loading and Preparation (continued) ---
import torch
import multiprocessing as mp

def collate_simple(batch):
    # batch is a list of dicts from __getitem__
    # Stack numeric fields; keep strings as Python lists.
    e        = torch.stack([b["e"] for b in batch], dim=0)
    y        = torch.tensor([float(b["y"]) for b in batch], dtype=torch.float32)
    id_idx   = torch.tensor([int(b["id_idx"]) for b in batch], dtype=torch.int64)
    is_real  = torch.tensor([int(b["is_real"]) for b in batch], dtype=torch.int64)

    identity   = [b["identity"]   for b in batch]   # leave as list[str]
    segment_id = [b["segment_id"] for b in batch]   # leave as list[str]

    return {
        "e": e, "y": y, "id_idx": id_idx, "is_real": is_real,
        "identity": identity, "segment_id": segment_id
    }

# Detect GPU availability
HAS_CUDA = torch.cuda.is_available()

# Optional: safer multiprocessing start for notebooks
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass  # it's already set

# Choose sensible defaults
if HAS_CUDA:
    DEVICE = torch.device("cuda")
    # Set NUM_WORKERS to 0 explicitly to avoid PicklingError in Colab
    NUM_WORKERS = 0
    PIN_MEMORY = True            # speeds up host→device transfers
else:
    DEVICE = torch.device("cpu")
    NUM_WORKERS = 0              # prevents multiprocessing errors in notebooks
    PIN_MEMORY = False

BATCH = 256

# We will explicitly set num_workers=0 in each DataLoader definition
# COMMON_KW = dict(
#     num_workers=NUM_WORKERS,
#     pin_memory=PIN_MEMORY,
#     collate_fn=collate_simple,
# )

In [4]:
# --- Data Loading and Preparation (continued) ---
openl3_data = np.load('/content/drive/MyDrive/Project_4_Deepfake_Detection/Fall 2025/Embeddings/111k_October_1/openl3_2025-09-12_audio_none.npz', allow_pickle=True)
openl3_data.files
#

['embeddings',
 'labels',
 'segment_ids',
 'model_name',
 'version',
 'mode',
 'noise',
 'denoiser_name']

In [5]:
# --- Data Loading and Preparation (continued) ---
print(openl3_data.files)

['embeddings', 'labels', 'segment_ids', 'model_name', 'version', 'mode', 'noise', 'denoiser_name']


In [6]:
# --- Data Loading and Preparation (continued) ---
# Check the length of 'embeddings' and 'labels'
embeddings_length = len(openl3_data['embeddings'])
labels_length = len(openl3_data['labels'])

print(f"Length of embeddings: {embeddings_length}")
print(f"Length of labels: {labels_length}")

# Get value counts of 'labels'
labels_value_counts = pd.Series(openl3_data['labels']).value_counts()

print("\nValue counts of labels:")
print(labels_value_counts)

Length of embeddings: 111003
Length of labels: 111003

Value counts of labels:
0    80402
1    30601
Name: count, dtype: int64


In [7]:
# --- Data Loading and Preparation (continued) ---
openl3_df = pd.DataFrame({
    'embedding': [row for row in openl3_data['embeddings']],
    'label': openl3_data['labels'],
    'segment_id': openl3_data['segment_ids']
})

print(openl3_df.head())
print(type(openl3_df['embedding'].iloc[0]))  # should be <class 'numpy.ndarray'>
print(openl3_df['embedding'].iloc[0].shape)  # should be (512,)

                                           embedding  label  \
0  [4.3699307, 1.8775293, 3.0148158, 0.5936537, 2...      1   
1  [4.794632, 2.0156565, 3.7857802, 0.5123592, 3....      1   
2  [4.228516, 2.079344, 3.2617736, 0.5693688, 3.6...      1   
3  [4.2838717, 2.202094, 3.3074424, 0.34179068, 3...      1   
4  [4.1662717, 1.9869436, 3.2502947, 0.9076028, 3...      1   

                                  segment_id  
0  02502690-d80f-594c-85e8-922f2e2ebde1_0000  
1  02502690-d80f-594c-85e8-922f2e2ebde1_0001  
2  02502690-d80f-594c-85e8-922f2e2ebde1_0002  
3  02502690-d80f-594c-85e8-922f2e2ebde1_0003  
4  02502690-d80f-594c-85e8-922f2e2ebde1_0004  
<class 'numpy.ndarray'>
(512,)


In [8]:
# --- Data Loading and Preparation (continued) ---
# create a column called 'identity'. In order to get the value of this, you should crop segment_id to the first 2 components of it: 0N1oA9LUEc4/00008/00008_1120_1440   vs 0N1oA9LUEc4/00008
# Split by '/' and take the first two components, then join them back
def extract_identity(x):
    if '/' in x:
        # if slashes exist, take the first two components
        return '/'.join(x.split('/')[:2])
    else:
        # otherwise, just remove the last underscore component
        return '_'.join(x.split('_')[:-1])

openl3_df['identity'] = openl3_df['segment_id'].apply(extract_identity)
openl3_df['label'] = 1 - openl3_df['label'] # Assuming original label was 1 for fake, 0 for real. Flipping to 0 for fake, 1 for real.
# Check result
print(openl3_df[['segment_id', 'identity', 'label']].head(10)) # Include label in check

                                  segment_id  \
0  02502690-d80f-594c-85e8-922f2e2ebde1_0000   
1  02502690-d80f-594c-85e8-922f2e2ebde1_0001   
2  02502690-d80f-594c-85e8-922f2e2ebde1_0002   
3  02502690-d80f-594c-85e8-922f2e2ebde1_0003   
4  02502690-d80f-594c-85e8-922f2e2ebde1_0004   
5  02502690-d80f-594c-85e8-922f2e2ebde1_0005   
6  02502690-d80f-594c-85e8-922f2e2ebde1_0006   
7  02502690-d80f-594c-85e8-922f2e2ebde1_0007   
8  02502690-d80f-594c-85e8-922f2e2ebde1_0008   
9  02502690-d80f-594c-85e8-922f2e2ebde1_0009   

                               identity  label  
0  02502690-d80f-594c-85e8-922f2e2ebde1      0  
1  02502690-d80f-594c-85e8-922f2e2ebde1      0  
2  02502690-d80f-594c-85e8-922f2e2ebde1      0  
3  02502690-d80f-594c-85e8-922f2e2ebde1      0  
4  02502690-d80f-594c-85e8-922f2e2ebde1      0  
5  02502690-d80f-594c-85e8-922f2e2ebde1      0  
6  02502690-d80f-594c-85e8-922f2e2ebde1      0  
7  02502690-d80f-594c-85e8-922f2e2ebde1      0  
8  02502690-d80f-594c-85e8-922

In [9]:
# --- Data Loading and Preparation (continued) ---
#### Analysis

import pandas as pd

# Count how many rows each identity has
id_counts = openl3_df['identity'].value_counts().rename_axis('identity').reset_index(name='count')

# Compute descriptive statistics
print("📊 Descriptive stats on #segments per identity:")
print(id_counts['count'].describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]))

# Merge back into the main df to compute real/fake breakdowns
if 'label' in openl3_df.columns:  # assuming 0=fake, 1=real
    breakdown = openl3_df.groupby(['identity', 'label']).size().unstack(fill_value=0)
    breakdown['total'] = breakdown.sum(axis=1)
    # Ensure columns 0 and 1 exist before calculating ratios
    breakdown['real_ratio'] = breakdown.get(1, 0) / breakdown['total']
    breakdown['fake_ratio'] = breakdown.get(0, 0) / breakdown['total']


    # Show top few identities by total count
    print("\n🧩 Sample breakdown per identity (top 10):")
    print(breakdown.sort_values('total', ascending=False).head(10))

    # And summary stats over the real/fake ratio distribution
    print("\n📈 Real ratio stats across identities:")
    print(breakdown['real_ratio'].describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]))
else:
    print("⚠️ No 'label' column found, skipping real/fake breakdown.")

# Optionally, you can merge the counts + ratios for export:
if 'label' in openl3_df.columns:
    stats_df = id_counts.merge(breakdown, left_on='identity', right_index=True)
else:
    stats_df = id_counts

# Save for inspection
stats_df.to_csv("identity_summary.csv", index=False)
print("\n✅ Saved detailed per-identity summary to identity_summary.csv")

📊 Descriptive stats on #segments per identity:
count    4096.000000
mean       27.100342
std        10.899494
min         3.000000
5%         10.000000
10%        15.000000
25%        25.000000
50%        25.000000
75%        32.000000
90%        39.000000
95%        50.000000
max        85.000000
Name: count, dtype: float64

🧩 Sample breakdown per identity (top 10):
label               0   1  total  real_ratio  fake_ratio
identity                                                
gMNlRovT5x4/00007  40  45     85    0.529412    0.470588
gMNlRovT5x4/00013  40  45     85    0.529412    0.470588
gMNlRovT5x4/00021  40  45     85    0.529412    0.470588
OkKhI6St40w/00016  40  45     85    0.529412    0.470588
gIuhIo7mO5A/00010  40  45     85    0.529412    0.470588
0OkOQhXhsIE/00009  40  45     85    0.529412    0.470588
NgbqXsA62Qs/00023  40  45     85    0.529412    0.470588
63tVsJI31Tk/00004  40  45     85    0.529412    0.470588
ORBf73HiJns/00029  40  45     85    0.529412    0.470588
VmP

In [10]:
# --- Data Loading and Preparation (continued) ---
print(len(openl3_df))
print(openl3_df['identity'].nunique())

111003
4096


In [11]:
# --- Data Loading and Preparation (continued) ---
print(openl3_df['label'].value_counts())

label
1    80402
0    30601
Name: count, dtype: int64


In [12]:
# --- Data Loading and Preparation (continued) ---
openl3_df.head()

,embedding,label,segment_id,identity
0,"[4.3699307, 1.8775293, 3.0148158, 0.5936537, 2...",0,02502690-d80f-594c-85e8-922f2e2ebde1_0000,02502690-d80f-594c-85e8-922f2e2ebde1
1,"[4.794632, 2.0156565, 3.7857802, 0.5123592, 3....",0,02502690-d80f-594c-85e8-922f2e2ebde1_0001,02502690-d80f-594c-85e8-922f2e2ebde1
2,"[4.228516, 2.079344, 3.2617736, 0.5693688, 3.6...",0,02502690-d80f-594c-85e8-922f2e2ebde1_0002,02502690-d80f-594c-85e8-922f2e2ebde1
3,"[4.2838717, 2.202094, 3.3074424, 0.34179068, 3...",0,02502690-d80f-594c-85e8-922f2e2ebde1_0003,02502690-d80f-594c-85e8-922f2e2ebde1
4,"[4.1662717, 1.9869436, 3.2502947, 0.9076028, 3...",0,02502690-d80f-594c-85e8-922f2e2ebde1_0004,02502690-d80f-594c-85e8-922f2e2ebde1


In [13]:
!pip -q install torch torchvision


In [14]:
# --- Data Loading and Preparation (continued) ---
### Dataset/Dataloader

import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.model_selection import train_test_split
import random
from collections import defaultdict
import numpy as np # Ensure numpy is imported

# ---------- 0) Prep ----------
df = openl3_df.copy()  # columns: embedding, label (1=real,0=fake), identity, segment_id
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("=== PREP ===")
print(f"Total rows: {len(df):,}")
if 'label' in df.columns:
    print("Label counts (1=real, 0=fake):")
    print(df['label'].value_counts(dropna=False).to_string())
else:
    print("WARNING: 'label' column not found.")

if 'identity' in df.columns:
    print(f"Unique identities: {df['identity'].nunique():,}")
else:
    print("WARNING: 'identity' column not found.")

# map identity -> int (handy for losses/eval)
id2idx = {s:i for i,s in enumerate(sorted(df['identity'].unique()))}
df['id_idx'] = df['identity'].map(id2idx).astype(np.int64)

# Try to infer embedding dim from first row safely
try:
    d_probe = len(df.iloc[0]['embedding'])
except Exception as e:
    d_probe = None
print(f"Inferred embedding dim (from first row): {d_probe}")

=== PREP ===
Total rows: 111,003
Label counts (1=real, 0=fake):
label
1    80402
0    30601
Unique identities: 4,096
Inferred embedding dim (from first row): 512


In [15]:
# --- Data Loading and Preparation (continued) ---
# ---------- 1) 70/15/15 split (standard Track A: identities may overlap) ----------
n = len(df)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

train_df, hold_df = train_test_split(
    df, test_size=(n - n_train), random_state=123, stratify=df['label']
)
val_df, test_df   = train_test_split(
    hold_df, test_size=(len(hold_df) - n_val), random_state=123, stratify=hold_df['label']
)

print("\n=== SPLITS (70/15/15) ===")
for name, x in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5}: {len(x):7,} rows | ids={x['identity'].nunique():5,}")
    print("       label counts:", x['label'].value_counts().to_dict())

# ---------- 1b) OPTIONAL: balance val/test (downsample reals to match fakes) ----------
def balanced_copy(x: pd.DataFrame) -> pd.DataFrame:
    pos = x[x['label'] == 1]
    neg = x[x['label'] == 0]
    if len(pos) == 0 or len(neg) == 0:
        print("  [balance] Skipping (one class empty).")
        return x.reset_index(drop=True)
    if len(pos) > len(neg):
        pos = pos.sample(n=len(neg), random_state=123)
    else:
        neg = neg.sample(n=len(pos), random_state=123)
    out = pd.concat([pos, neg]).sample(frac=1.0, random_state=123).reset_index(drop=True)
    return out

val_bal  = balanced_copy(val_df)
test_bal = balanced_copy(test_df)

print("\n=== AFTER BALANCING val/test ===")
for name, x in [("val_bal", val_bal), ("test_bal", test_bal)]:
    print(f"{name:>8}: {len(x):7,} rows | ids={x['identity'].nunique():5,}")
    print("          label counts:", x['label'].value_counts().to_dict())


=== SPLITS (70/15/15) ===
train:  77,702 rows | ids=4,096
       label counts: {1: 56281, 0: 21421}
  val:  16,650 rows | ids=3,926
       label counts: {1: 12060, 0: 4590}
 test:  16,651 rows | ids=3,927
       label counts: {1: 12061, 0: 4590}

=== AFTER BALANCING val/test ===
 val_bal:   9,180 rows | ids=3,225
          label counts: {1: 4590, 0: 4590}
test_bal:   9,180 rows | ids=3,245
          label counts: {1: 4590, 0: 4590}


In [16]:
# --- Data Loading and Preparation (continued) ---
# ---------- 2) OPTIONAL Track B: identity-disjoint test for z_het ----------
# Keep train/val from above; create a special "unseen ID" test set:
ids_all = df['identity'].unique()
ids_train_seen = set(train_df['identity'].unique())
ids_unseen = np.array([i for i in ids_all if i not in ids_train_seen])

if len(ids_unseen) > 0:
    print(f"\n=== UNSEEN-ID POOL ===")
    print("# of unseen identities available:", len(ids_unseen))
    pick = min(600, len(ids_unseen))
    unseen_ids = set(np.random.RandomState(123).choice(ids_unseen, size=pick, replace=False))
    zhet_test_df = df[df['identity'].isin(unseen_ids)].reset_index(drop=True)
    print(f"zhet_test_df: {len(zhet_test_df):,} rows | ids={zhet_test_df['identity'].nunique():,}")
else:
    print("\n(No unseen identities left for z_het test set.)")
    zhet_test_df = pd.DataFrame(columns=df.columns)  # empty fallback


(No unseen identities left for z_het test set.)


In [17]:
# --- Data Loading and Preparation (continued) ---
# ---------- 3) Dataset ----------
class EmbDataset(Dataset):
    def __init__(self, frame_df):
        df = frame_df.reset_index(drop=True)

        # Pre-extract arrays/lists to avoid .iloc each time
        self.emb = [np.asarray(x, dtype=np.float32) for x in df['embedding'].tolist()]
        self.y   = df['label'].astype(np.float32).to_numpy()
        self.idi = df['id_idx'].astype(np.int64).to_numpy()
        self.real= (df['label'] == 1).astype(np.int64).to_numpy()
        self.ident = df['identity'].tolist()
        self.segid = df['segment_id'].tolist()

        # infer embedding dim
        self.d = int(self.emb[0].shape[0])

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return {
            "e": torch.from_numpy(self.emb[i]),
            "y": torch.tensor(self.y[i]),
            "id_idx": torch.tensor(self.idi[i]),
            "is_real": torch.tensor(self.real[i]),
            "identity": self.ident[i],
            "segment_id": self.segid[i],
        }

train_ds   = EmbDataset(train_df)
val_ds     = EmbDataset(val_bal)    # use val_df instead if you don't want balancing
test_ds    = EmbDataset(test_bal)
# guard: zhet_test_df might be empty
zhet_test_ds = EmbDataset(zhet_test_df) if len(zhet_test_df) > 0 else None

BATCH = 256

print("\n=== DATASET SUMMARY ===")
print(f"train_ds: {len(train_ds):,} rows | emb_dim={train_ds.d}")
print(f"  val_ds: {len(val_ds):,} rows | emb_dim={val_ds.d}")
print(f" test_ds: {len(test_ds):,} rows | emb_dim={test_ds.d}")
if zhet_test_ds is not None:
    print(f"zhet_test_ds: {len(zhet_test_ds):,} rows | emb_dim={zhet_test_ds.d}")
else:
    print("zhet_test_ds: None")


=== DATASET SUMMARY ===
train_ds: 77,702 rows | emb_dim=512
  val_ds: 9,180 rows | emb_dim=512
 test_ds: 9,180 rows | emb_dim=512
zhet_test_ds: None


In [18]:
# --- Data Loading and Preparation (continued) ---
# Explicitly setting num_workers=0 and pin_memory=True/False based on DEVICE
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE.type == 'cuda'),
                          collate_fn=collate_simple)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == 'cuda'),
                          collate_fn=collate_simple)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == 'cuda'),
                          collate_fn=collate_simple)

print(f"Using device: {DEVICE}, workers={0}, pin_memory={(DEVICE.type == 'cuda')}")

Using device: cuda, workers=0, pin_memory=True


In [19]:
# --- Data Loading and Preparation (continued) ---
# Quick single-item probe
print("train_ds len:", len(train_ds))
sample = train_ds[0]
print({k: (tuple(v.shape) if torch.is_tensor(v) else type(v)) for k,v in sample.items()})

train_ds len: 77702
{'e': (512,), 'y': (), 'id_idx': (), 'is_real': (), 'identity': <class 'str'>, 'segment_id': <class 'str'>}


In [20]:
# --- Data Loading and Preparation (continued) ---
# ---------- 4) Stage-A real-only, identity-balanced loader for L_het ----------
# Keep only identities with >=2 real samples
real_train = train_df[train_df['label'] == 1].copy()
counts = real_train.groupby('identity').size()
eligible_ids = set(counts[counts >= 2].index)

print("\n=== REAL-ONLY (for L_het) ===")
print(f"real_train rows: {len(real_train):,}  | identities (>=1 real): {real_train['identity'].nunique():,}")
print(f"eligible_ids (>=2 reals): {len(eligible_ids):,}")
if len(eligible_ids) < 10:
    print("WARNING: Very few eligible identities for SupCon. Consider smaller k or larger batch.")

class RealOnlyIDDataset(Dataset):
    def __init__(self, real_df, eligible_ids):
        subset = real_df[real_df['identity'].isin(eligible_ids)]
        assert len(subset) > 0, "RealOnlyIDDataset is empty. Check eligibility filter."
        self.df = subset.reset_index(drop=True)
        self.d  = int(len(self.df.iloc[0]['embedding']))
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        e = np.asarray(r['embedding'], dtype=np.float32)
        return {
            "e": torch.from_numpy(e),
            "y": torch.tensor(np.float32(r['label'])), # Include 'y'
            "id_idx": torch.tensor(np.int64(id2idx[r['identity']])),
            "identity": r['identity'],
            "is_real": torch.tensor(np.int64(r['label'] == 1)), # Include 'is_real'
            "segment_id": r['segment_id'], # Include 'segment_id'
        }

real_id_ds = RealOnlyIDDataset(real_train, eligible_ids)
print(f"real_id_ds: {len(real_id_ds):,} rows | emb_dim={real_id_ds.d}")

class IdentityBalancedBatchSampler(Sampler):
    """
    Yields batches with m identities x k samples each (reals only).
    """
    def __init__(self, dataset, m=32, k=4, seed=123):
        self.m, self.k = m, k
        self.by_id = defaultdict(list)
        for i in range(len(dataset.df)):
            ident = dataset.df.iloc[i]['identity']
            self.by_id[ident].append(i)
        self.ids = [i for i,v in self.by_id.items() if len(v) >= k]
        self.rng = random.Random(seed)
        # debug
        lens = [len(v) for v in self.by_id.values()]
        print("\n=== ID-BALANCED SAMPLER ===")
        print(f"m={m}, k={k}, candidate_ids (>=k): {len(self.ids):,} / total_ids: {len(self.by_id):,}")
        print(f"Per-id sample count (min/mean/max): {min(lens)} / {np.mean(lens):.1f} / {max(lens)}")

    def __iter__(self):
        ids = self.ids[:]
        self.rng.shuffle(ids)
        for s in range(0, len(ids), self.m):
            chunk = ids[s:s+self.m]
            batch = []
            for ident in chunk:
                idxs = self.rng.sample(self.by_id[ident], self.k)
                batch.extend(idxs)
            yield batch

    def __len__(self):
        # approximate number of batches
        return max(1, len(self.ids) // self.m)

M, K = 32, 4  # batch = M*K
real_id_loader = DataLoader(
    real_id_ds,
    batch_sampler=IdentityBalancedBatchSampler(real_id_ds, m=M, k=K),
    num_workers=0, # Explicitly set to 0
    pin_memory=(DEVICE.type == 'cuda'),
    collate_fn=collate_simple,
)

# ---------- Peek at a few batches to confirm shapes ----------
def peek_loader(name, loader, take=1):
    print(f"\n>>> Peeking {name} (first {take} batch[es])")
    try:
        it = iter(loader)
    except Exception as e:
        print("  failed to create iterator:", repr(e)); return

    for b in range(take):
        try:
            batch = next(it)
        except StopIteration:
            print("  (empty)"); return
        except Exception as e:
            print("  next() failed:", repr(e)); return
        keys = {k: (tuple(v.shape) if torch.is_tensor(v) else type(v))
                for k, v in batch.items() if k in ["e","y","id_idx","is_real"]}
        print("  keys/shapes:", keys)
        if "identity" in batch:
            ids = batch["identity"]
            try:
                uniq = len(set(ids if isinstance(ids, list) else list(ids)))
            except Exception:
                uniq = "n/a"
            print(f"  identities in batch: {uniq}")


peek_loader("train_loader", train_loader, take=1)
peek_loader("val_loader",   val_loader,   take=1)
peek_loader("test_loader",  test_loader,  take=1)
peek_loader("real_id_loader (for L_het)", real_id_loader, take=1)

# embedding dimension (assume consistent with train set)
d = train_ds.d
print(f"\nFinal embedding dimension d = {d}")


=== REAL-ONLY (for L_het) ===
real_train rows: 56,281  | identities (>=1 real): 3,421
eligible_ids (>=2 reals): 3,421
real_id_ds: 56,281 rows | emb_dim=512

=== ID-BALANCED SAMPLER ===
m=32, k=4, candidate_ids (>=k): 3,373 / total_ids: 3,421
Per-id sample count (min/mean/max): 2 / 16.5 / 38

>>> Peeking train_loader (first 1 batch[es])
  keys/shapes: {'e': (256, 512), 'y': (256,), 'id_idx': (256,), 'is_real': (256,)}
  identities in batch: 252

>>> Peeking val_loader (first 1 batch[es])
  keys/shapes: {'e': (256, 512), 'y': (256,), 'id_idx': (256,), 'is_real': (256,)}
  identities in batch: 248

>>> Peeking test_loader (first 1 batch[es])
  keys/shapes: {'e': (256, 512), 'y': (256,), 'id_idx': (256,), 'is_real': (256,)}
  identities in batch: 247

>>> Peeking real_id_loader (for L_het) (first 1 batch[es])
  keys/shapes: {'e': (128, 512), 'y': (128,), 'id_idx': (128,), 'is_real': (128,)}
  identities in batch: 32

Final embedding dimension d = 512


# Model Definitions



*   **Adapter**: Tiny residual MLP initialized to identity. It slightly reshapes or fine-tinues input embeddings *e*
*   **f_hom: (first MLP HEAD)**: Projects embeddings nto real manifold z_hom. *f_hom(e) = z_hom*
*   **f_id: (second MLP HEAD)**: Projects embeddings into identity/heterogeneous manifold z_het. *f_het(e) = z_het*
*   **Frame Classifiers**: Predicts real/fake label from z_hom (supervised classifier head with L_cls)

*   **Stage A (warm-up)**: Train only Adapter, f_hom, f_id using L_hom + L_orth

*   **Stage B (supervised)**: Add FrameClassifier on top oif f_hom and L_cls




In [21]:
# --- Model Definitions ---
# 2) Model: Adapter + Heads + Classifier

import torch.nn as nn
import torch.nn.functional as F
import torch # Ensure torch is imported

class Adapter(nn.Module):
    """Tiny residual 'pseudo-backbone' to lightly reshape frozen embeddings."""
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, d)
        self.act = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(d, d)
        # init near identity
        nn.init.zeros_(self.fc1.weight); nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.weight); nn.init.zeros_(self.fc2.bias)
    def forward(self, e):
        return e + self.fc2(self.act(self.fc1(e)))

class MLPHead(nn.Module):
    def __init__(self, d_in, d_out=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256), nn.ReLU(inplace=True),
            nn.Linear(256, d_out)
        )
    def forward(self, x):
        z = self.net(x)
        return F.normalize(z, dim=-1)

class FrameClassifier(nn.Module):
    def __init__(self, d_in=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128), nn.ReLU(inplace=True),
            nn.Linear(128, 1)
        )
    def forward(self, z):
        return self.net(z).squeeze(-1)   # logits

# Define the original model components
# Assuming 'd' (embedding dimension) is defined earlier
adapter = Adapter(d).to(DEVICE)
f_hom   = MLPHead(d, 128).to(DEVICE)
f_id    = MLPHead(d, 128).to(DEVICE)
clf     = FrameClassifier(128).to(DEVICE)


# --- Ablation Study Models ---

# Experiment 1a: Linear Probe
class LinearProbe(nn.Module):
    """Simple linear classifier on raw embeddings."""
    def __init__(self, d_in):
        super().__init__()
        self.linear = nn.Linear(d_in, 1)

    def forward(self, e):
        # Output logits directly
        return self.linear(e).squeeze(-1)

# Experiment 2: Stage B on its own (Adapter + MLPHead + FrameClassifier trained directly)
class DirectClassifier(nn.Module):
    """Adapter + MLPHead + FrameClassifier trained directly on raw embeddings."""
    def __init__(self, d_in, d_hidden=128):
        super().__init__()
        self.adapter = Adapter(d_in) # Use the Adapter class
        self.mlp_head = MLPHead(d_in, d_hidden) # Use the MLPHead class
        self.classifier = FrameClassifier(d_hidden) # Use the FrameClassifier class

    def forward(self, e):
        e_adapted = self.adapter(e)
        z = self.mlp_head(e_adapted) # z is the output of the MLPHead (equivalent to z_hom's dimension)
        logits = self.classifier(z) # Pass z to the FrameClassifier
        return logits # logits

# Loss Functions and Helpers

In [22]:
## Loss helpers

def cov(feats):  # feats: [N, C]
    x = feats - feats.mean(0, keepdim=True)
    return (x.T @ x) / (max(1, len(feats)-1))

def offdiag_frobenius(M):
    return (M - torch.diag(torch.diag(M))).pow(2).sum()

def variance_compactness(z_reals):
    # L_hom: keep real z_deepfake compact (VICReg-style)
    if z_reals.shape[0] <= 1:
        return z_reals.sum() * 0.0
    C = cov(z_reals)
    return torch.trace(C)

def cross_cov_penalty(z_a, z_b):
    # L_orth: decorrelate the two heads
    za = z_a - z_a.mean(0, keepdim=True)
    zb = z_b - z_b.mean(0, keepdim=True)
    C = (za.T @ zb) / (max(1, len(za)-1))
    return offdiag_frobenius(C)


In [23]:
# --- Loss Functions and Helpers (continued) ---
## Supcon Loss Helpers

import torch.nn.functional as F # Ensure F is imported
import random # Ensure random is imported

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    def forward(self, anchor, positives, negatives):
        device = anchor.device
        B = anchor.shape[0]
        anchor = F.normalize(anchor, dim=1)
        positives = F.normalize(positives, dim=2)
        negatives = F.normalize(negatives, dim=2)
        pos_sim = torch.bmm(anchor.unsqueeze(1), positives.transpose(1,2)).squeeze(1) / self.temperature
        neg_sim = torch.bmm(anchor.unsqueeze(1), negatives.transpose(1,2)).squeeze(1) / self.temperature
        sims = torch.cat([pos_sim, neg_sim], dim=1)
        labels = torch.zeros(B, sims.size(1), device=device)
        labels[:, :pos_sim.size(1)] = 1
        exp_sims = torch.exp(sims)
        log_prob = sims - torch.log(exp_sims.sum(dim=1, keepdim=True) + 1e-12)
        mean_log_pos = (labels * log_prob).sum(1) / (labels.sum(1) + 1e-12)
        return -mean_log_pos.mean()

def build_pos_neg_from_batch(z_id_real, id_idx_real, max_pos=4, max_neg=32):
    """
    z_id_real: [Br, D] (only real rows of current batch)
    id_idx_real: [Br]  int labels for identity (only real rows)
    Returns anchors [B',D], positives [B',P,D], negatives [B',N,D]
    Only anchors with >=1 positive are kept.
    """
    device = z_id_real.device
    Br, D = z_id_real.shape
    if Br < 2:
        return None  # not enough reals to form pairs

    anchors, pos_list, neg_list = [], [], []
    ids = id_idx_real

    # Precompute indices per identity
    by_id = {}
    for i in range(Br):
        by_id.setdefault(int(ids[i].item()), []).append(i)

    for i in range(Br):
        my_id = int(ids[i].item())
        same = by_id[my_id].copy()
        if i in same:
            same.remove(i)
        diff = [j for j in range(Br) if int(ids[j].item()) != my_id]
        if len(same) == 0 or len(diff) == 0:
            continue  # skip anchors without any positive or negative

        p_idx = torch.tensor(random_sample(same, k=max_pos), device=device)
        n_idx = torch.tensor(random_sample(diff, k=max_neg), device=device)

        anchors.append(z_id_real[i])
        pos_list.append(z_id_real[p_idx])
        neg_list.append(z_id_real[n_idx])

    if len(anchors) == 0:
        return None

    A = torch.stack(anchors, dim=0)                           # [B', D]
    P = torch.stack([p for p in pos_list], dim=0)             # [B', P, D]
    N = torch.stack([n for n in neg_list], dim=0)             # [B', N, D]
    return A, P, N

def random_sample(lst, k):
    # sample up to k with replacement if needed
    if len(lst) >= k:
        import random
        return random.sample(lst, k)
    # pad by repeating if too few
    import numpy as np
    idx = lst.copy()
    while len(idx) < k:
        idx += lst
    return idx[:k]

# Training Loops

In [24]:
# --- Training Loops ---
from tqdm.auto import tqdm # Ensure tqdm is imported
import torch # Ensure torch is imported
import torch.nn as nn # Ensure nn is imported
import numpy as np # Ensure numpy is imported


# Simple training loop for the linear probe (reusing BCE loss from main code)
def train_linear_probe(loader, model, optimizer, criterion, epochs=5, log_every=100):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        n = 0
        pbar = tqdm(loader, desc=f"[Linear Probe Epoch {epoch+1}]", leave=False)
        for it, batch in enumerate(pbar, 1):
            e = batch["e"].to(DEVICE)
            y = batch["y"].to(DEVICE)

            optimizer.zero_grad()
            logits = model(e)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            bs = e.size(0); n += bs
            running_loss += loss.item() * bs

            if it % log_every == 0:
                pbar.set_postfix({"loss": f"{running_loss/max(1,n):.4f}"})

        print(f"[Linear Probe Epoch {epoch+1}] Avg Loss: {running_loss/max(1,n):.4f}")


# Training loop for the linear probe on z_hom
def train_linear_probe_zhom(loader, adapter_m, f_hom_m, linear_probe_m, optimizer_m, criterion, epochs=5, log_every=100):
    adapter_m.eval(); f_hom_m.eval() # Freeze adapter and f_hom
    linear_probe_m.train() # Train only the linear probe

    for epoch in range(epochs):
        running_loss = 0.0
        n = 0
        pbar = tqdm(loader, desc=f"[LP on z_hom Epoch {epoch+1}]", leave=False)
        for it, batch in enumerate(pbar, 1):
            e = batch["e"].to(DEVICE)
            y = batch["y"].to(DEVICE)

            with torch.no_grad(): # Ensure no gradients flow back to adapter/f_hom
                e2 = adapter_m(e)
                z_hom = f_hom_m(e2)

            optimizer_m.zero_grad()
            logits = linear_probe_m(z_hom)
            loss = criterion(logits, y)
            loss.backward()
            optimizer_m.step()

            bs = e.size(0); n += bs
            running_loss += loss.item() * bs

            if it % log_every == 0:
                pbar.set_postfix({"loss": f"{running_loss/max(1,n):.4f}"})

        print(f"[LP on z_hom Epoch {epoch+1}] Avg Loss: {running_loss/max(1,n):.4f}")

# Refactored Stage A training function for experiments
def train_stageA_experiment(loader, adapter_m, f_hom_m, f_id_m, optimizer_m,
                            orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                            W_HOM=1.0, W_ORTH=0.1, W_HET_A=0.2):
    adapter_m.train(); f_hom_m.train(); f_id_m.train()
    running_loss = {"hom":0.0, "orth":0.0, "het":0.0, "total":0.0}; n=0
    supcon_loss_fn = SupConLoss()

    pbar = tqdm(loader, desc=f"[Stage A Exp (W_ORTH={W_ORTH})]", leave=False)
    for it, batch in enumerate(pbar, 1):
        e  = batch["e"].to(DEVICE)
        mr = batch["is_real"].to(DEVICE).bool()
        ids= batch["id_idx"].to(DEVICE)

        e2    = adapter_m(e)
        z_hom = f_hom_m(e2)
        z_idv = f_id_m(e2)

        L_hom = variance_compactness(z_hom[mr]) if mr.any() else torch.tensor(0.0, device=DEVICE)

        L_orth = torch.tensor(0.0, device=DEVICE)
        if W_ORTH > 0: # Calculate L_orth only if W_ORTH is positive
            zha, zia = (z_hom[mr], z_idv[mr]) if orth_on_reals_only and mr.any() else (z_hom, z_idv)
            L_orth = cross_cov_penalty(zha, zia) if zha.shape[0] > 1 else torch.tensor(0.0, device=DEVICE)

        supcon_data = build_pos_neg_from_batch(z_idv[mr], ids[mr], max_pos=max_pos, max_neg=max_neg)
        L_het = supcon_loss_fn(*supcon_data) if supcon_data else torch.tensor(0.0, device=DEVICE)

        loss = W_HOM * L_hom + W_ORTH * L_orth + W_HET_A * L_het

        optimizer_m.zero_grad()
        loss.backward()
        optimizer_m.step()

        bs = e.size(0); n += bs
        running_loss["hom"]  += L_hom.item() * bs
        running_loss["orth"] += L_orth.item() * bs
        running_loss["het"]  += L_het.item() * bs
        running_loss["total"]+= loss.item() * bs

        if it % log_every == 0:
             pbar.set_postfix({k: f"{v/max(1,n):.4f}" for k,v in running_loss.items()})

    for k in running_loss: running_loss[k] /= max(1, n)
    return running_loss

# Refactored Stage B training function for experiments
def train_stageB_experiment(loader, adapter_m, f_hom_m, f_id_m, clf_m, optimizer_m,
                            keep_supcon=False, orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                            W_HOM=1.0, W_ORTH=0.1, W_HET_B=0.0, W_CLS=1.0):
    adapter_m.train(); f_hom_m.train(); f_id_m.train(); clf_m.train()
    running_loss = {"hom":0.0, "orth":0.0, "het":0.0, "cls":0.0, "total":0.0}; n=0
    supcon_loss_fn = SupConLoss()
    bce_loss_fn = nn.BCEWithLogitsLoss()

    pbar = tqdm(loader, desc=f"[Stage B Exp (W_ORTH={W_ORTH})]", leave=False)
    for it, batch in enumerate(pbar, 1):
        e  = batch["e"].to(DEVICE)
        y  = batch["y"].to(DEVICE)
        mr = batch["is_real"].to(DEVICE).bool()
        ids= batch["id_idx"].to(DEVICE)

        e2    = adapter_m(e)
        z_hom = f_hom_m(e2)
        z_idv = f_id_m(e2)
        logits= clf_m(z_hom)

        L_hom = variance_compactness(z_hom[mr]) if mr.any() else torch.tensor(0.0, device=DEVICE)

        L_orth = torch.tensor(0.0, device=DEVICE)
        if W_ORTH > 0: # Calculate L_orth only if W_ORTH is positive
            zha, zia = (z_hom[mr], z_idv[mr]) if orth_on_reals_only and mr.any() else (z_hom, z_idv)
            L_orth = cross_cov_penalty(zha, zia) if zha.shape[0] > 1 else torch.tensor(0.0, device=DEVICE)

        L_cls = bce_loss_fn(logits, y)

        L_het = torch.tensor(0.0, device=DEVICE)
        if keep_supcon:
             supcon_data = build_pos_neg_from_batch(z_idv[mr], ids[mr], max_pos=max_pos, max_neg=max_neg)
             L_het = supcon_loss_fn(*supcon_data) if supcon_data else torch.tensor(0.0, device=DEVICE)


        loss = W_HOM * L_hom + W_ORTH * L_orth + W_HET_B * L_het + W_CLS * L_cls

        optimizer_m.zero_grad()
        loss.backward()
        optimizer_m.step()

        bs = e.size(0); n += bs
        running_loss["hom"]  += L_hom.item() * bs
        running_loss["orth"] += L_orth.item() * bs
        running_loss["het"]  += L_het.item() * bs
        running_loss["cls"]  += L_cls.item() * bs
        running_loss["total"]+= loss.item() * bs

        if it % log_every == 0:
             pbar.set_postfix({k: f"{v/max(1,n):.4f}" for k,v in running_loss.items()})

    # Average losses over the epoch
    for k in running_loss: running_loss[k] /= max(1, n)
    return running_loss

# Training loop for the DirectClassifier
def train_direct_classifier(loader, model, optimizer, criterion, epochs=10, log_every=100): # More epochs for direct training
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        n = 0
        pbar = tqdm(loader, desc=f"[Direct CLS Epoch {epoch+1}]", leave=False)
        for it, batch in enumerate(pbar, 1):
            e = batch["e"].to(DEVICE)
            y = batch["y"].to(DEVICE)

            optimizer.zero_grad()
            logits = model(e)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            bs = e.size(0); n += bs
            running_loss += loss.item() * bs

            if it % log_every == 0:
                pbar.set_postfix({"loss": f"{running_loss/max(1,n):.4f}"})

        print(f"[Direct CLS Epoch {epoch+1}] Avg Loss: {running_loss/max(1,n):.4f}")

# Training loop for the linear probe on z_hom
def train_linear_probe_zhom(loader, adapter_m, f_hom_m, linear_probe_m, optimizer_m, criterion, epochs=5, log_every=100):
    adapter_m.eval(); f_hom_m.eval() # Freeze adapter and f_hom
    linear_probe_m.train() # Train only the linear probe

    for epoch in range(epochs):
        running_loss = 0.0
        n = 0
        pbar = tqdm(loader, desc=f"[LP on z_hom Epoch {epoch+1}]", leave=False)
        for it, batch in enumerate(pbar, 1):
            e = batch["e"].to(DEVICE)
            y = batch["y"].to(DEVICE)

            with torch.no_grad(): # Ensure no gradients flow back to adapter/f_hom
                e2 = adapter_m(e)
                z_hom = f_hom_m(e2)

            optimizer_m.zero_grad()
            logits = linear_probe_m(z_hom)
            loss = criterion(logits, y)
            loss.backward()
            optimizer_m.step()

            bs = e.size(0); n += bs
            running_loss += loss.item() * bs

            if it % log_every == 0:
                pbar.set_postfix({"loss": f"{running_loss/max(1,n):.4f}"})

        print(f"[LP on z_hom Epoch {epoch+1}] Avg Loss: {running_loss/max(1,n):.4f}")

# Modified Stage A training function without the adapter
def train_stageA_noadapter(loader, f_hom_m, f_id_m, optimizer_m,
                           orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                           W_HOM=1.0, W_ORTH=0.1, W_HET_A=0.2):
    f_hom_m.train(); f_id_m.train()
    running_loss = {"hom":0.0, "orth":0.0, "het":0.0, "total":0.0}; n=0
    supcon_loss_fn = SupConLoss()

    pbar = tqdm(loader, desc=f"[Stage A NoAdapter (W_ORTH={W_ORTH})]", leave=False)
    for it, batch in enumerate(pbar, 1):
        e  = batch["e"].to(DEVICE) # Use raw embedding directly

        mr = batch["is_real"].to(DEVICE).bool()
        ids= batch["id_idx"].to(DEVICE)

        # forward (skip adapter)
        z_hom = f_hom_m(e) # Pass raw embedding 'e' directly to f_hom
        z_idv = f_id_m(e) # Pass raw embedding 'e' directly to f_id

        # --- L_hom on REALS only ---
        L_hom = variance_compactness(z_hom[mr]) if mr.any() else torch.tensor(0.0, device=DEVICE)

        # --- L_orth (your call: real-only or whole batch) ---
        L_orth = torch.tensor(0.0, device=DEVICE)
        if W_ORTH > 0:
            zha, zia = (z_hom[mr], z_idv[mr]) if orth_on_reals_only and mr.any() else (z_hom, z_idv)
            L_orth = cross_cov_penalty(zha, zia) if zha.shape[0] > 1 else torch.tensor(0.0, device=DEVICE)

        # --- L_het (SupCon on REALS only) ---
        supcon_data = build_pos_neg_from_batch(z_idv[mr], ids[mr], max_pos=max_pos, max_neg=max_neg)
        L_het = supcon_loss_fn(*supcon_data) if supcon_data else torch.tensor(0.0, device=DEVICE)

        loss = W_HOM * L_hom + W_ORTH * L_orth + W_HET_A * L_het

        optimizer_m.zero_grad()
        loss.backward()
        optimizer_m.step()

        bs = e.size(0); n += bs
        running_loss["hom"]  += L_hom.item() * bs
        running_loss["orth"] += L_orth.item() * bs
        running_loss["het"]  += L_het.item() * bs
        running_loss["total"]+= loss.item() * bs

        if it % log_every == 0:
             pbar.set_postfix({k: f"{v/max(1,n):.4f}" for k,v in running_loss.items()})

    for k in running_loss: running_loss[k] /= max(1, n)
    return running_loss

# Modified Stage B training function without the adapter
def train_stageB_noadapter(loader, f_hom_m, f_id_m, clf_m, optimizer_m,
                           keep_supcon=False, orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                           W_HOM=1.0, W_ORTH=0.1, W_HET_B=0.0, W_CLS=1.0):
    f_hom_m.train(); f_id_m.train(); clf_m.train()
    running_loss = {"hom":0.0, "orth":0.0, "het":0.0, "cls":0.0, "total":0.0}; n=0
    supcon_loss_fn = SupConLoss()
    bce_loss_fn = nn.BCEWithLogitsLoss()

    pbar = tqdm(loader, desc=f"[Stage B NoAdapter (W_ORTH={W_ORTH})]", leave=False)
    for it, batch in enumerate(pbar, 1):
        e  = batch["e"].to(DEVICE) # Use raw embedding directly
        y  = batch["y"].to(DEVICE)
        mr = batch["is_real"].to(DEVICE).bool()
        ids= batch["id_idx"].to(DEVICE)

        # forward (skip adapter)
        z_hom = f_hom_m(e) # Pass raw embedding 'e' directly to f_hom
        z_idv = f_id_m(e) # Pass raw embedding 'e' directly to f_id
        logits= clf_m(z_hom)

        L_hom = variance_compactness(z_hom[mr]) if mr.any() else torch.tensor(0.0, device=DEVICE)

        L_orth = torch.tensor(0.0, device=DEVICE)
        if W_ORTH > 0:
            zha, zia = (z_hom[mr], z_idv[mr]) if orth_on_reals_only and mr.any() else (z_hom, z_idv)
            L_orth = cross_cov_penalty(zha, zia) if zha.shape[0] > 1 else torch.tensor(0.0, device=DEVICE)

        L_cls = bce_loss_fn(logits, y)

        L_het = torch.tensor(0.0, device=DEVICE)
        if keep_supcon:
             supcon_data = build_pos_neg_from_batch(z_idv[mr], ids[mr], max_pos=max_pos, max_neg=max_neg)
             L_het = supcon_loss_fn(*supcon_data) if supcon_data else torch.tensor(0.0, device=DEVICE)


        loss = W_HOM * L_hom + W_ORTH * L_orth + W_HET_B * L_het + W_CLS * L_cls

        optimizer_m.zero_grad()
        loss.backward()
        optimizer_m.step()

        bs = e.size(0); n += bs
        running_loss["hom"]  += L_hom.item() * bs
        running_loss["orth"] += L_orth.item() * bs
        running_loss["het"]  += L_het.item() * bs
        running_loss["cls"]  += L_cls.item() * bs
        running_loss["total"]+= loss.item() * bs

        if it % log_every == 0:
             pbar.set_postfix({k: f"{v/max(1,n):.4f}" for k,v in running_loss.items()})

    for k in running_loss: running_loss[k] /= max(1, n)
    return running_loss

# Modified Stage A training function without the adapter
def train_stageA_noadapter(loader, f_hom_m, f_id_m, optimizer_m,
                           orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                           W_HOM=1.0, W_ORTH=0.1, W_HET_A=0.2):
    f_hom_m.train(); f_id_m.train()
    running_loss = {"hom":0.0, "orth":0.0, "het":0.0, "total":0.0}; n=0
    supcon_loss_fn = SupConLoss()

    pbar = tqdm(loader, desc=f"[Stage A NoAdapter (W_ORTH={W_ORTH})]", leave=False)
    for it, batch in enumerate(pbar, 1):
        e  = batch["e"].to(DEVICE) # Use raw embedding directly

        mr = batch["is_real"].to(DEVICE).bool()
        ids= batch["id_idx"].to(DEVICE)

        # forward (skip adapter)
        z_hom = f_hom_m(e) # Pass raw embedding 'e' directly to f_hom
        z_idv = f_id_m(e) # Pass raw embedding 'e' directly to f_id

        # --- L_hom on REALS only ---
        L_hom = variance_compactness(z_hom[mr]) if mr.any() else torch.tensor(0.0, device=DEVICE)

        # --- L_orth (your call: real-only or whole batch) ---
        L_orth = torch.tensor(0.0, device=DEVICE)
        if W_ORTH > 0:
            zha, zia = (z_hom[mr], z_idv[mr]) if orth_on_reals_only and mr.any() else (z_hom, z_idv)
            L_orth = cross_cov_penalty(zha, zia) if zha.shape[0] > 1 else torch.tensor(0.0, device=DEVICE)

        # --- L_het (SupCon on REALS only) ---
        supcon_data = build_pos_neg_from_batch(z_idv[mr], ids[mr], max_pos=max_pos, max_neg=max_neg)
        L_het = supcon_loss_fn(*supcon_data) if supcon_data else torch.tensor(0.0, device=DEVICE)

        loss = W_HOM * L_hom + W_ORTH * L_orth + W_HET_A * L_het

        optimizer_m.zero_grad()
        loss.backward()
        optimizer_m.step()

        bs = e.size(0); n += bs
        running_loss["hom"]  += L_hom.item() * bs
        running_loss["orth"] += L_orth.item() * bs
        running_loss["het"]  += L_het.item() * bs
        running_loss["total"]+= loss.item() * bs

        if it % log_every == 0:
             pbar.set_postfix({k: f"{v/max(1,n):.4f}" for k,v in running_loss.items()})

    for k in running_loss: running_loss[k] /= max(1, n)
    return running_loss


# Modified Stage B training function without the adapter
def train_stageB_noadapter(loader, f_hom_m, f_id_m, clf_m, optimizer_m,
                           keep_supcon=False, orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                           W_HOM=1.0, W_ORTH=0.1, W_HET_B=0.0, W_CLS=1.0):
    f_hom_m.train(); f_id_m.train(); clf_m.train()
    running_loss = {"hom":0.0, "orth":0.0, "het":0.0, "cls":0.0, "total":0.0}; n=0
    supcon_loss_fn = SupConLoss()
    bce_loss_fn = nn.BCEWithLogitsLoss()

    pbar = tqdm(loader, desc=f"[Stage B NoAdapter (W_ORTH={W_ORTH})]", leave=False)
    for it, batch in enumerate(pbar, 1):
        e  = batch["e"].to(DEVICE) # Use raw embedding directly
        y  = batch["y"].to(DEVICE)
        mr = batch["is_real"].to(DEVICE).bool()
        ids= batch["id_idx"].to(DEVICE)

        # forward (skip adapter)
        z_hom = f_hom_m(e) # Pass raw embedding 'e' directly to f_hom
        z_idv = f_id_m(e) # Pass raw embedding 'e' directly to f_id
        logits= clf_m(z_hom)

        L_hom = variance_compactness(z_hom[mr]) if mr.any() else torch.tensor(0.0, device=DEVICE)

        L_orth = torch.tensor(0.0, device=DEVICE)
        if W_ORTH > 0:
            zha, zia = (z_hom[mr], z_idv[mr]) if orth_on_reals_only and mr.any() else (z_hom, z_idv)
            L_orth = cross_cov_penalty(zha, zia) if zha.shape[0] > 1 else torch.tensor(0.0, device=DEVICE)

        L_cls = bce_loss_fn(logits, y)

        L_het = torch.tensor(0.0, device=DEVICE)
        if keep_supcon:
             supcon_data = build_pos_neg_from_batch(z_idv[mr], ids[mr], max_pos=max_pos, max_neg=max_neg)
             L_het = supcon_loss_fn(*supcon_data) if supcon_data else torch.tensor(0.0, device=DEVICE)


        loss = W_HOM * L_hom + W_ORTH * L_orth + W_HET_B * L_het + W_CLS * L_cls

        optimizer_m.zero_grad()
        loss.backward()
        optimizer_m.step()

        bs = e.size(0); n += bs
        running_loss["hom"]  += L_hom.item() * bs
        running_loss["orth"] += L_orth.item() * bs
        running_loss["het"]  += L_het.item() * bs
        running_loss["cls"]  += L_cls.item() * bs
        running_loss["total"]+= loss.item() * bs

        if it % log_every == 0:
             pbar.set_postfix({k: f"{v/max(1,n):.4f}" for k,v in running_loss.items()})

    for k in running_loss: running_loss[k] /= max(1, n)
    return running_loss


In [25]:
# Optimizers: (optionally) exclude clf in Stage-A
# Assuming 'd' (embedding dimension) is defined earlier
opt_stageA = torch.optim.AdamW(
    list(adapter.parameters()) + list(f_hom.parameters()) + list(f_id.parameters()),
    lr=1e-3, weight_decay=1e-4
)
opt_stageB = torch.optim.AdamW(
    list(adapter.parameters()) + list(f_hom.parameters()) + list(f_id.parameters()) + list(clf.parameters()),
    lr=1e-3, weight_decay=1e-4
)

bce = nn.BCEWithLogitsLoss()
supcon = SupConLoss(temperature=0.07).to(DEVICE)

# weights you can tune
W_HOM   = 0.2
W_ORTH  = 0.1
W_HET_A = 0.5   # Stage-A weight for SupCon
W_HET_B = 0.25  # Stage-B (optional) weight if you keep it on

# Evaluation Functions

**EVALUATION**


*   **GOAL:** Test f_hom
*   **Method:** How far is this embedding from the real manifold. If we look at distribution of embeddings from the real samples, we want to see how far any new embedding (real or fake) is from that distribution.
*   **Metric:** Mahalanobis Distance. Measures distance to a distribution, not just a point. Accounts for feature/dimension covariance (correlation and scale). Answers: "How anomalous is this sample relative to the real embedding distribution?"





In [28]:
# --- Evaluation Functions ---
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm # Ensure tqdm is imported
import numpy as np # Ensure numpy is imported
import torch.nn.functional as F # Ensure F is imported
import torch # Ensure torch is imported


### Testing f_hom
### Fit on training REALS only as calibartion test
@torch.no_grad()
def fit_real_gaussian(loader):
    adapter.eval(); f_hom.eval()
    reals = []
    for b in loader:
        e = b["e"].to(DEVICE)
        is_real = b["is_real"].to(DEVICE).bool()
        # Ensure adapter and f_hom are used as in the original model
        z = f_hom(adapter(e))[is_real]
        if z.numel(): reals.append(z.cpu())
    if not reals:
        print("\nWarning: No real samples found in loader for fitting Gaussian.")
        # Return dummy values or raise error if appropriate for downstream use
        # Assuming 'd' is the embedding dimension
        dummy_mu = torch.zeros(f_hom(adapter(torch.zeros(1, d).to(DEVICE))).shape[-1])
        dummy_inv_cov = torch.eye(dummy_mu.shape[-1])
        return dummy_mu, dummy_inv_cov


    Z = torch.cat(reals, dim=0)
    mu = Z.mean(0)
    X  = Z - mu
    C  = (X.T @ X) / max(1, len(Z)-1)
    # Add small diagonal for numerical stability, size should match C
    C  = C + 1e-3 * torch.eye(C.size(0), device=C.device)
    inv_cov = torch.linalg.inv(C)
    return mu, inv_cov

@torch.no_grad()
def mahalanobis(z, mu, inv_cov):
    diff = z - mu.to(z.device)
    # Use matmul instead of einsum for clarity and potential performance
    # Ensure inv_cov is on the correct device
    m2 = torch.sum((diff @ inv_cov.to(z.device)) * diff, dim=1)
    return torch.sqrt(torch.clamp(m2, min=0))

@torch.no_grad()
def eval_distance_auc(loader, mu, inv_cov):
    adapter.eval(); f_hom.eval()
    ys, ds = [], []
    pbar = tqdm(loader, desc="[Eval Mahalanobis]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        # Ensure adapter and f_hom are used as in the original model
        z = f_hom(adapter(e))
        d = mahalanobis(z, mu, inv_cov).cpu().numpy()
        ys.append(y); ds.append(d)
    ys = np.concatenate(ys); ds = np.concatenate(ds)

    # For Mahalanobis distance, lower distance means more likely "real" (label 1).
    # roc_auc_score expects higher scores for the positive class.
    # To use raw distance, flip the labels: 0 becomes 1 (fake), 1 becomes 0 (real).
    if len(np.unique(ys)) < 2:
         print("\nWarning: Only one class present in evaluation labels for Mahalanobis AUC.")
         return float('nan')

    return roc_auc_score(1 - ys, ds) # Flip true labels here


@torch.no_grad()
def fit_raw_real_gaussian(loader):
    """Fits a Gaussian on raw real embeddings from the training set."""
    reals = []
    pbar = tqdm(loader, desc="[Fit Raw Real Gaussian]", leave=False)
    for b in pbar:
        e = b["e"].to(DEVICE)
        is_real = b["is_real"].to(DEVICE).bool()
        raw_reals = e[is_real]
        if raw_reals.numel(): reals.append(raw_reals.cpu())

    if not reals:
        print("\nWarning: No real samples found in loader for fitting Raw Gaussian.")
        dummy_mu = torch.zeros(d)
        dummy_inv_cov = torch.eye(d)
        return dummy_mu, dummy_inv_cov

    Z = torch.cat(reals, dim=0)
    mu = Z.mean(0)
    X  = Z - mu
    C  = (X.T @ X) / max(1, len(Z)-1)
    C  = C + 1e-3 * torch.eye(C.size(0), device=C.device)
    inv_cov = torch.linalg.inv(C)
    return mu, inv_cov

@torch.no_grad()
def eval_raw_distance_auc(loader, mu, inv_cov):
    """Evaluates Mahalanobis AUC on raw embeddings."""
    ys, ds = [], []
    pbar = tqdm(loader, desc="[Eval Raw Mahalanobis]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        d = mahalanobis(e, mu, inv_cov).cpu().numpy() # Use raw embedding 'e'
        ys.append(y); ds.append(d)
    ys = np.concatenate(ys); ds = np.concatenate(ds)

    if len(np.unique(ys)) < 2:
         print("\nWarning: Only one class present in evaluation labels for Raw Mahalanobis AUC.")
         return float('nan')

    # Flip labels as higher distance means more likely fake (label 0)
    return roc_auc_score(1 - ys, ds)

@torch.no_grad()
def embed_features(loader, embedding_key='z_het', model_components=None):
    """
    Collects embeddings ('e' or 'z_het') and associated metadata.
    model_components: tuple (adapter, f_hom, f_id) to allow evaluation of specific model states.
    """
    # Use provided model components or globals if none provided
    adapter_eval = model_components[0] if model_components else adapter
    f_hom_eval = model_components[1] if model_components else f_hom
    f_id_eval = model_components[2] if model_components else f_id

    adapter_eval.eval(); f_hom_eval.eval(); f_id_eval.eval() # Ensure models are in eval mode

    Embeddings, ID, IS_REAL = [], [], []
    pbar = tqdm(loader, desc=f"[Collecting {embedding_key} embeddings]", leave=False)
    for b in pbar:
        e = b["e"].to(DEVICE)
        if embedding_key == 'z_het':
            # Ensure adapter and f_id are used for z_het
            z = F.normalize(f_id_eval(adapter_eval(e)), dim=-1)
            Embeddings.append(z.cpu())
        elif embedding_key == 'e':
            Embeddings.append(e.cpu()) # Use raw embedding 'e'
        elif embedding_key == 'z_hom':
             # Ensure adapter and f_hom are used for z_hom
             z = f_hom_eval(adapter_eval(e))
             Embeddings.append(z.cpu())
        else:
            raise ValueError(f"Unknown embedding_key: {embedding_key}")

        ID.append(b["id_idx"].cpu())
        IS_REAL.append(b["is_real"].cpu())

    return torch.cat(Embeddings), torch.cat(ID), torch.cat(IS_REAL)

# Evaluation function for the Linear Probe
@torch.no_grad()
def eval_linear_probe_auc(loader, model):
    """Evaluates classification AUC for the linear probe."""
    model.eval()
    ys, logits = [], []
    pbar = tqdm(loader, desc="[Eval Linear Probe]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        logit = model(e).cpu().numpy()
        ys.append(y); logits.append(logit)
    ys = np.concatenate(ys); logits = np.concatenate(logits)

    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for Linear Probe AUC.")
        return float('nan')

    probs = 1 / (1 + np.exp(-logits)) # sigmoid manually
    auc = roc_auc_score(ys, probs)
    return auc

# Evaluation function for the linear probe on z_hom
@torch.no_grad()
def eval_linear_probe_zhom_auc(loader, adapter_m, f_hom_m, linear_probe_m):
    adapter_m.eval(); f_hom_m.eval(); linear_probe_m.eval()
    ys, logits = [], []
    pbar = tqdm(loader, desc="[Eval LP on z_hom]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        with torch.no_grad():
            e2 = adapter_m(e)
            z_hom = f_hom_m(e2)
        logit = linear_probe_m(z_hom).cpu().numpy()
        ys.append(y); logits.append(logit)
    ys = np.concatenate(ys); logits = np.concatenate(logits)

    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for LP on z_hom AUC.")
        return float('nan')

    probs = 1 / (1 + np.exp(-logits)) # sigmoid manually
    auc = roc_auc_score(ys, probs)
    return auc

@torch.no_grad()
def knn_id_acc(train_loader, test_loader, embedding_key='z_het', k=1, model_components=None):
    """
    KNN identity accuracy using specified embeddings.
    gallery = train reals; queries = test reals
    model_components: tuple (adapter, f_hom, f_id) to allow evaluation of specific model states.
    """
    print(f"\n--- Evaluating KNN Identity Accuracy on {embedding_key} embeddings (k={k}) ---")
    # Collect embeddings for gallery (train reals) and queries (test reals)
    Z_tr, ID_tr, R_tr = embed_features(train_loader, embedding_key=embedding_key, model_components=model_components)
    Z_te, ID_te, R_te = embed_features(test_loader, embedding_key=embedding_key, model_components=model_components)


    gallery_Z  = Z_tr[R_tr.bool()]
    gallery_ID = ID_tr[R_tr.bool()]
    query_Z    = Z_te[R_te.bool()]
    query_ID   = ID_te[R_te.bool()]

    if len(gallery_Z) == 0 or len(query_Z) == 0:
        print(f"Warning: Not enough real samples in train or test for KNN evaluation on {embedding_key}.")
        return float('nan')

    # Compute cosine similarity between query embeddings and gallery embeddings
    # Normalize embeddings first for cosine similarity
    gallery_Z_norm = F.normalize(gallery_Z.float(), dim=1)
    query_Z_norm = F.normalize(query_Z.float(), dim=1)

    sims = query_Z_norm @ gallery_Z_norm.T

    # Find k nearest neighbors and predict identity
    if k == 1:
        pred = gallery_ID[sims.argmax(dim=1)]
    else:
        topk = sims.topk(k=k, dim=1).indices
        pred = torch.mode(gallery_ID[topk], dim=1).values

    # Calculate accuracy
    acc = (pred == query_ID).float().mean().item()
    return acc


# Evaluation function for classification AUC and BCE loss
@torch.no_grad()
def eval_epoch_cls(loader, model_components=None):
    """Evaluates classification AUC and BCE loss."""
    # Use provided model components or globals if none provided
    adapter_eval = model_components[0] if model_components else adapter
    f_hom_eval = model_components[1] if model_components else f_hom
    clf_eval = model_components[3] if model_components and len(model_components) > 3 else clf

    # Conditionally set to eval mode
    if adapter_eval: adapter_eval.eval()
    if f_hom_eval: f_hom_eval.eval()
    if clf_eval: clf_eval.eval()


    total_loss = 0.0
    n = 0
    all_logits, all_y = [], []
    bce_loss_fn = nn.BCEWithLogitsLoss() # Corrected typo: BCEWithItsLoss -> BCEWithLogitsLoss


    pbar = tqdm(loader, desc="[Eval CLS]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE); y = batch["y"].to(DEVICE)

        # Handle case where adapter or f_hom might be None in model_components
        if adapter_eval and f_hom_eval:
            z_hom = f_hom_eval(adapter_eval(e))
        elif f_hom_eval: # Case without adapter
            z_hom = f_hom_eval(e)
        else:
            print("Warning: Cannot evaluate classification, f_hom_eval not available.")
            # Return dummy values or skip evaluation if needed
            return float('nan'), float('nan') # Or handle as an error


        if clf_eval:
            logit = clf_eval(z_hom)
            loss = bce_loss_fn(logit, y)
            total_loss += loss.item()*e.size(0); n += e.size(0)
            all_logits.append(logit.sigmoid().cpu())
            all_y.append(y.cpu())
            pbar.set_postfix({"bce": f"{total_loss/max(1,n):.4f}"})
        else:
             print("Warning: Cannot evaluate classification, clf_eval not available.")
             return float('nan'), float('nan')


    import numpy as np
    from sklearn.metrics import roc_auc_score
    y_true = torch.cat(all_y).numpy()
    y_hat  = torch.cat(all_logits).numpy()
    auc = roc_auc_score(y_true, y_hat) if len(np.unique(y_true))>1 else float('nan')
    return total_loss/max(1,n), auc


# Evaluation function for the DirectClassifier (similar to eval_epoch_cls but for DirectClassifier)
@torch.no_grad()
def eval_direct_classifier_auc(loader, model):
    model.eval()
    ys, logits = [], []
    pbar = tqdm(loader, desc="[Eval Direct CLS]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        logit = model(e).cpu().numpy()
        ys.append(y); logits.append(logit)
    ys = np.concatenate(ys); logits = np.concatenate(logits)

    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for Direct Classifier AUC.")
        return float('nan')

    probs = 1 / (1 + np.exp(-logits)) # sigmoid manually
    auc = roc_auc_score(ys, probs)
    return auc

# Mahalanobis on f_hom output (trained without adapter)
@torch.no_grad()
def fit_real_gaussian_noadapter(loader, f_hom_m):
    f_hom_m.eval()
    reals = []
    pbar = tqdm(loader, desc="[Fit Real Gaussian NoAdapter]", leave=False)
    for b in pbar:
        e = b["e"].to(DEVICE) # Use raw embedding
        is_real = b["is_real"].to(DEVICE).bool()
        z = f_hom_m(e)[is_real] # Use f_hom directly on raw embedding
        if z.numel(): reals.append(z.cpu())
    if not reals:
        print("\nWarning: No real samples found in loader for fitting Gaussian (No Adapter).")
        dummy_mu = torch.zeros(f_hom_m(torch.zeros(1, d).to(DEVICE)).shape[-1])
        dummy_inv_cov = torch.eye(dummy_mu.shape[-1])
        return dummy_mu, dummy_inv_cov
    Z = torch.cat(reals, dim=0)
    mu = Z.mean(0)
    X  = Z - mu
    C  = (X.T @ X) / max(1, len(Z)-1)
    C  = C + 1e-3 * torch.eye(C.size(0), device=C.device)
    inv_cov = torch.linalg.inv(C)
    return mu, inv_cov

@torch.no_grad()
def eval_distance_auc_noadapter(loader, f_hom_m, mu, inv_cov):
    f_hom_m.eval()
    ys, ds = [], []
    pbar = tqdm(loader, desc="[Eval Mahalanobis NoAdapter]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE) # Use raw embedding
        y = batch["y"].cpu().numpy()
        z = f_hom_m(e) # Use f_hom directly on raw embedding
        d = mahalanobis(z, mu, inv_cov).cpu().numpy()
        ys.append(y); ds.append(d)
    ys = np.concatenate(ys); ds = np.concatenate(ds)
    if len(np.unique(ys)) < 2:
         print("\nWarning: Only one class present in evaluation labels for Mahalanobis AUC (No Adapter).")
         return float('nan')
    return roc_auc_score(1 - ys, ds) # Flip true labels

# KNN Identity on f_id output (trained without adapter)
# Reuse knn_id_acc but pass the f_id model and specify embedding_key='z_het' (even though it's not normalized here)
# Need a modified embed_features or knn_id_acc to handle no adapter
@torch.no_grad()
def embed_features_noadapter(loader, f_hom_m=None, f_id_m=None, embedding_key='z_het'):
    """Collects embeddings ('e', f_hom(e) or f_id(e)) and associated metadata, no adapter."""
    if f_hom_m: f_hom_m.eval()
    if f_id_m: f_id_m.eval()

    Embeddings, ID, IS_REAL = [], [], []
    pbar = tqdm(loader, desc=f"[Collecting {embedding_key} embeddings NoAdapter]", leave=False)
    for b in pbar:
        e = b["e"].to(DEVICE) # Use raw embedding directly

        if embedding_key == 'z_het' and f_id_m:
            z = F.normalize(f_id_m(e), dim=-1) # Use f_id directly on raw embedding
            Embeddings.append(z.cpu())
        elif embedding_key == 'z_hom' and f_hom_m:
             z = f_hom_m(e) # Use f_hom directly on raw embedding
             Embeddings.append(z.cpu())
        elif embedding_key == 'e':
            Embeddings.append(e.cpu()) # Use raw embedding 'e'
        else:
            raise ValueError(f"Unknown embedding_key: {embedding_key} or missing model.")

        ID.append(b["id_idx"].cpu())
        IS_REAL.append(b["is_real"].cpu())

    return torch.cat(Embeddings), torch.cat(ID), torch.cat(IS_REAL)

@torch.no_grad()
def knn_id_acc_noadapter(train_loader, test_loader, f_id_m, embedding_key='z_het', k=1):
    """
    KNN identity accuracy using specified embeddings (no adapter).
    gallery = train reals; queries = test reals
    """
    print(f"\n--- Evaluating KNN Identity Accuracy on {embedding_key} embeddings (No Adapter, k={k}) ---")
    # Collect embeddings for gallery (train reals) and queries (test reals)
    Z_tr, ID_tr, R_tr = embed_features_noadapter(train_loader, f_id_m=f_id_m, embedding_key=embedding_key)
    Z_te, ID_te, R_te = embed_features_noadapter(test_loader, f_id_m=f_id_m, embedding_key=embedding_key)


    gallery_Z  = Z_tr[R_tr.bool()]
    gallery_ID = ID_tr[R_tr.bool()]
    query_Z    = Z_te[R_te.bool()]
    query_ID   = ID_te[R_te.bool()]

    if len(gallery_Z) == 0 or len(query_Z) == 0:
        print(f"Warning: Not enough real samples in train or test for KNN evaluation on {embedding_key} (No Adapter).")
        return float('nan')

    # Compute cosine similarity between query embeddings and gallery embeddings
    gallery_Z_norm = F.normalize(gallery_Z.float(), dim=1)
    query_Z_norm = F.normalize(query_Z.float(), dim=1)

    sims = query_Z_norm @ gallery_Z_norm.T

    # Find k nearest neighbors and predict identity
    if k == 1:
        pred = gallery_ID[sims.argmax(dim=1)]
    else:
        topk = sims.topk(k=k, dim=1).indices
        pred = torch.mode(gallery_ID[topk], dim=1).values

    # Calculate accuracy
    acc = (pred == query_ID).float().mean().item()
    return acc

# Classification AUC on f_hom->clf (trained without adapter)
# Need a modified eval_epoch_cls or specific calls
@torch.no_grad()
def eval_classification_auc_noadapter(loader, f_hom_m, clf_m):
    f_hom_m.eval(); clf_m.eval()
    ys, logits = [], []
    bce_loss_fn = nn.BCEWithLogitsLoss()
    total_loss = 0.0
    n = 0
    pbar = tqdm(loader, desc="[Eval CLS AUC NoAdapter]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE) # Use raw embedding
        y = batch["y"].to(DEVICE)
        z_hom = f_hom_m(e) # Use f_hom directly on raw embedding
        logit = clf_m(z_hom)
        loss = bce_loss_fn(logit, y)
        total_loss += loss.item()*e.size(0); n += e.size(0)
        ys.append(y.cpu().numpy()); logits.append(logit.sigmoid().cpu().numpy())
    ys = np.concatenate(ys); logits = np.concatenate(logits)
    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for Classification AUC (No Adapter).")
        return total_loss/max(1,n), float('nan')
    auc = roc_auc_score(ys, logits)
    return total_loss/max(1,n), auc


def run_all_evals(stage_name="Current Stage", model_components=None):
    """
    Runs a standard set of evaluations.
    model_components: tuple (adapter, f_hom, f_id, clf) to allow evaluation of specific model states.
    """
    print(f"\n=== EVALUATION: {stage_name} ===")

    # Mahalanobis (fit on TRAIN reals) - uses z_hom
    # Need to use model components consistently if provided
    mu, inv_cov = fit_real_gaussian(train_loader) # fit_real_gaussian uses adapter and f_hom
    val_auc_maha  = eval_distance_auc(val_loader,  mu, inv_cov) # eval_distance_auc uses adapter and f_hom
    test_auc_maha = eval_distance_auc(test_loader, mu, inv_cov)
    print(f"[Eval Mahalanobis z_hom]  val_auc={val_auc_maha:.3f}  test_auc={test_auc_maha:.3f}")

    # Classification AUC (uses z_hom and clf)
    # Pass model components to eval_epoch_cls if they are provided
    print("\n--- Evaluating Classification AUC ---")
    # Check if clf is available in model_components or globals and trainable
    if (model_components and len(model_components) > 3) or ('clf' in globals() and any(p.requires_grad for p in clf.parameters())):
         val_bce, val_auc_cls = eval_epoch_cls(val_loader, model_components=model_components)
         test_bce, test_auc_cls = eval_epoch_cls(test_loader, model_components=model_components) # Added test evaluation
         print(f"[Eval Classification z_hom->clf] val_auc={val_auc_cls:.3f}  test_auc={test_auc_cls:.3f}")
         print(f"[Eval Classification z_hom->clf] test_bce={test_bce:.4f}  test_auc={test_auc_cls:.3f}") # Print test BCE
    else:
        print("Skipping Classification AUC: Classifier not available or not trainable.")


    # KNN identity on z_het (gallery=train reals, queries=test reals)
    # Uses the consolidated knn_id_acc function, pass model components if provided
    acc1_zhet = knn_id_acc(train_loader, test_loader, embedding_key='z_het', k=1, model_components=model_components)
    acc5_zhet = knn_id_acc(train_loader, test_loader, embedding_key='z_het', k=5, model_components=model_components)
    print(f"[Eval KNN z_het]  test@1={acc1_zhet:.3f}  test@5={acc5_zhet:.3f}")

# Placeholder for the original run_all_evals call at the end of training cell
# This will be moved to the final combined execution cell later.
# print("\n=== FINAL EVAL AFTER STAGE B ===")
# run_all_evals("Final Model")

# Ablation Studies

In [29]:
# --- Ablation Studies ---

print("--- Running Ablation Studies ---")

# Store results in a dictionary
results = {}

# Experiment 1: Raw Embedding Analysis

print("\n--- Experiment 1: Raw Embedding Analysis ---")

# 1a) Linear Probe for Classification AUC on Raw Embeddings

# Define the linear probe model (defined in Model Definitions section)
linear_probe_model = LinearProbe(d).to(DEVICE)

# Define optimizer for the linear probe (Assuming bce is defined in Loss Functions section)
opt_linear_probe = torch.optim.AdamW(linear_probe_model.parameters(), lr=1e-3, weight_decay=1e-4)

print("\nTraining Linear Probe...")
# Ensure bce loss is defined (it should be in the Loss Functions section)
bce = nn.BCEWithLogitsLoss()
train_linear_probe(train_loader, linear_probe_model, opt_linear_probe, bce, epochs=5)

print("\nEvaluating Linear Probe AUC...")
# Ensure eval_linear_probe_auc is defined in Evaluation Functions section
val_lp_auc = eval_linear_probe_auc(val_loader, linear_probe_model)
test_lp_auc = eval_linear_probe_auc(test_loader, linear_probe_model)
print(f"[Raw Emb Linear Probe AUC] val_auc={val_lp_auc:.3f}  test_auc={test_lp_auc:.3f}")
results['Experiment 1a (Linear Probe on Raw E)'] = {'val_auc_cls': val_lp_auc, 'test_auc_cls': test_lp_auc}


# 1b) KNN Identity Accuracy on Raw Embeddings
print("\nEvaluating Raw Embedding KNN Identity...")
# Use the consolidated knn_id_acc function (defined in Evaluation Functions section) with embedding_key='e'
raw_knn_acc1 = knn_id_acc(train_loader, test_loader, embedding_key='e', k=1)
raw_knn_acc5 = knn_id_acc(train_loader, test_loader, embedding_key='e', k=5)
print(f"[Raw Emb KNN Identity] test@1={raw_knn_acc1:.3f}  test@5={raw_knn_acc5:.3f}")
results['Experiment 1b (KNN on Raw E)'] = {'test_knn_1': raw_knn_acc1, 'test_knn_5': raw_knn_acc5}

# 1c) Mahalanobis AUC on Raw Embeddings
print("\nEvaluating Mahalanobis AUC on Raw Embeddings...")


# Fit Gaussian on raw real train embeddings
mu_raw, inv_cov_raw = fit_raw_real_gaussian(train_loader)

# Evaluate Mahalanobis AUC on val and test raw embeddings
val_auc_maha_raw  = eval_raw_distance_auc(val_loader,  mu_raw, inv_cov_raw)
test_auc_maha_raw = eval_raw_distance_auc(test_loader, mu_raw, inv_cov_raw)
print(f"[Raw Emb Mahalanobis AUC] val_auc={val_auc_maha_raw:.3f}  test_auc={test_auc_maha_raw:.3f}")
results['Experiment 1c (Mahalanobis on Raw E)'] = {'val_auc_maha': val_auc_maha_raw, 'test_auc_maha': test_auc_maha_raw}


# Experiment 2: Stage B on its own (DirectClassifier on Input Embeddings)

print("\n--- Experiment 2: Stage B on its own (Direct Classifier) ---")

# Define the DirectClassifier model (defined in Model Definitions section)
direct_classifier_model = DirectClassifier(d).to(DEVICE)
opt_direct_classifier = torch.optim.AdamW(direct_classifier_model.parameters(), lr=1e-4, weight_decay=1e-4) # Using a slightly lower LR as a starting point

print("\nTraining Direct Classifier...")
# Ensure train_direct_classifier is defined in Training Loops section
# bce is already defined
train_direct_classifier(train_loader, direct_classifier_model, opt_direct_classifier, bce, epochs=10)

print("\nEvaluating Direct Classifier AUC...")
# Ensure eval_direct_classifier_auc is defined in Evaluation Functions section
val_dc_auc = eval_direct_classifier_auc(val_loader, direct_classifier_model)
test_dc_auc = eval_direct_classifier_auc(test_loader, direct_classifier_model)
print(f"[Direct Classifier AUC] val_auc={val_dc_auc:.3f}  test_auc={test_dc_auc:.3f}")
results['Experiment 2 (Direct Classifier)'] = {'val_auc_cls': val_dc_auc, 'test_auc_cls': test_dc_auc}


# Experiment 3: Stage A on its own

print("\n--- Experiment 3: Stage A on its own ---")
print("Evaluating the model state after Stage A (Warm-up).")

# Re-initialize model for a clean Stage A run (Models defined in Model Definitions section)
adapter_exp3 = Adapter(d).to(DEVICE)
f_hom_exp3   = MLPHead(d, 128).to(DEVICE)
f_id_exp3    = MLPHead(d, 128).to(DEVICE)
clf_exp3     = FrameClassifier(128).to(DEVICE) # Include clf for run_all_evals structure, though not trained in Stage A

# Need optimizers for this specific run
opt_stageA_exp3 = torch.optim.AdamW(
    list(adapter_exp3.parameters()) + list(f_hom_exp3.parameters()) + list(f_id_exp3.parameters()),
    lr=1e-4, weight_decay=1e-4
)

# Train Stage A (using the experiment-specific training function defined in Training Loops section)
print("\nTraining Stage A for evaluation...")
for ep in range(3): # Same number of epochs as your original Stage A
    stats_exp3 = train_stageA_experiment(real_id_loader, adapter_exp3, f_hom_exp3, f_id_exp3, opt_stageA_exp3,
                                        orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                                        W_HOM=W_HOM, W_ORTH=W_ORTH, W_HET_A=W_HET_A) # W_HOM, W_ORTH, W_HET_A from Loss Functions section
    print(f"[Stage A Exp {ep+1}] " +
          f"hom={stats_exp3['hom']:.4f} orth={stats_exp3['orth']:.4f} het={stats_exp3['het']:.4f} total={stats_exp3['total']:.4f}")


# Evaluate after Stage A using the specific models from this experiment run (using run_all_evals defined in Evaluation Functions section)
exp3_eval_results = run_all_evals("Experiment 3: Stage A Only", model_components=(adapter_exp3, f_hom_exp3, f_id_exp3, clf_exp3)) # Pass clf_exp3 too for classification eval if needed
results['Experiment 3 (Stage A Only)'] = exp3_eval_results

# 3b) Linear Probe on z_hom after Stage A
print("\n--- Experiment 3b: Linear Probe on z_hom after Stage A ---")

# Define a new Linear Probe model specifically for z_hom (output of f_hom)
linear_probe_zhom = LinearProbe(128).to(DEVICE) # d_out of MLPHead is 128

# Define optimizer for this linear probe
opt_linear_probe_zhom = torch.optim.AdamW(linear_probe_zhom.parameters(), lr=1e-3, weight_decay=1e-4)




print("\nTraining Linear Probe on z_hom after Stage A...")
# Use the trained models from Experiment 3 (adapter_exp3, f_hom_exp3)
train_linear_probe_zhom(train_loader, adapter_exp3, f_hom_exp3, linear_probe_zhom, opt_linear_probe_zhom, bce, epochs=5)

print("\nEvaluating Linear Probe on z_hom AUC...")
val_lp_zhom_auc = eval_linear_probe_zhom_auc(val_loader, adapter_exp3, f_hom_exp3, linear_probe_zhom)
test_lp_zhom_auc = eval_linear_probe_zhom_auc(test_loader, adapter_exp3, f_hom_exp3, linear_probe_zhom)
print(f"[LP on z_hom AUC] val_auc={val_lp_zhom_auc:.3f}  test_auc={test_lp_zhom_auc:.3f}")
results['Experiment 3b (Linear Probe on z_hom Stage A)'] = {'val_auc_cls': val_lp_zhom_auc, 'test_auc_cls': test_lp_zhom_auc}


# Experiment 4: Stage A and B together (Original Model)

print("\n--- Experiment 4: Stage A and B Together (Original Model) ---")
print("Running the full two-stage training and evaluation.")

# Re-initialize original model components for a clean run (Models defined in Model Definitions section)
adapter_orig = Adapter(d).to(DEVICE)
f_hom_orig   = MLPHead(d, 128).to(DEVICE)
f_id_orig    = MLPHead(d, 128).to(DEVICE)
clf_orig     = FrameClassifier(128).to(DEVICE)

# Need optimizers for this specific run
opt_stageA_orig = torch.optim.AdamW(
    list(adapter_orig.parameters()) + list(f_hom_orig.parameters()) + list(f_id_orig.parameters()),
    lr=1e-4, weight_decay=1e-4
)
opt_stageB_orig = torch.optim.AdamW(
    list(adapter_orig.parameters()) + list(f_hom_orig.parameters()) + list(f_id_orig.parameters()) + list(clf_orig.parameters()),
    lr=1e-4, weight_decay=1e-4
)

# Run Stage A (using the experiment-specific training function defined in Training Loops section)
for ep in range(3):
    stats_stageA_orig = train_stageA_experiment(real_id_loader, adapter_orig, f_hom_orig, f_id_orig, opt_stageA_orig,
                                                orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                                                W_HOM=W_HOM, W_ORTH=W_ORTH, W_HET_A=W_HET_A) # W_HOM, W_ORTH, W_HET_A from Loss Functions section
    print(f"[Stage A Orig {ep+1}] " +
          f"hom={stats_stageA_orig['hom']:.4f} orth={stats_stageA_orig['orth']:.4f} het={stats_stageA_orig['het']:.4f} total={stats_stageA_orig['total']:.4f}")

# Run Stage B (using the experiment-specific training function defined in Training Loops section)
for ep in range(8):
    stats_stageB_orig = train_stageB_experiment(train_loader, adapter_orig, f_hom_orig, f_id_orig, clf_orig, opt_stageB_orig,
                                                keep_supcon=False, orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                                                W_HOM=W_HOM, W_ORTH=W_ORTH, W_HET_B=W_HET_B, W_CLS=1.0) # W_HOM, W_ORTH, W_HET_B, W_CLS from Loss Functions section

    # Evaluate classification after each Stage B epoch for monitoring (using eval_epoch_cls defined in Evaluation Functions section)
    val_bce_orig, val_auc_orig = eval_epoch_cls(val_loader, model_components=(adapter_orig, f_hom_orig, f_id_orig, clf_orig))
    print(f"[Stage B Orig {ep+1}] total={stats_stageB_orig['total']:.4f}  val_bce={val_bce_orig:.4f}  val_auc={val_auc_orig:.3f}")

# Final evaluation after Stage B (using run_all_evals defined in Evaluation Functions section)
exp4_eval_results = run_all_evals("Experiment 4: Stage A + B (Original)", model_components=(adapter_orig, f_hom_orig, f_id_orig, clf_orig))
results['Experiment 4 (Stage A + B)'] = exp4_eval_results


# Experiment 5: Stage A and B without Orthogonality Constraint

print("\n--- Experiment 5: Stage A + B without Orthogonality Constraint ---")

# Re-initialize model components for a clean run (Models defined in Model Definitions section)
adapter_noorth = Adapter(d).to(DEVICE)
f_hom_noorth   = MLPHead(d, 128).to(DEVICE)
f_id_noorth    = MLPHead(d, 128).to(DEVICE)
clf_noorth     = FrameClassifier(128).to(DEVICE)

# Need optimizers for this specific run
opt_stageA_noorth = torch.optim.AdamW(
    list(adapter_noorth.parameters()) + list(f_hom_noorth.parameters()) + list(f_id_noorth.parameters()),
    lr=1e-4, weight_decay=1e-4
)
opt_stageB_noorth = torch.optim.AdamW(
    list(adapter_noorth.parameters()) + list(f_hom_noorth.parameters()) + list(f_id_noorth.parameters()) + list(clf_noorth.parameters()),
    lr=1e-4, weight_decay=1e-4
)


# Run Stage A without Orthogonality (using the experiment-specific training function defined in Training Loops section)
for ep in range(3):
    stats_stageA_noorth = train_stageA_experiment(real_id_loader, adapter_noorth, f_hom_noorth, f_id_noorth, opt_stageA_noorth,
                                                     orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                                                     W_HOM=W_HOM, W_ORTH=0.0, W_HET_A=W_HET_A) # W_ORTH=0 here
    print(f"[Stage A NoOrth {ep+1}] " +
          f"hom={stats_stageA_noorth['hom']:.4f} het={stats_stageA_noorth['het']:.4f} total={stats_stageA_noorth['total']:.4f}")

# Run Stage B without Orthogonality (using the experiment-specific training function defined in Training Loops section)
for ep in range(8):
    stats_stageB_noorth = train_stageB_experiment(train_loader, adapter_noorth, f_hom_noorth, f_id_noorth, clf_noorth, opt_stageB_noorth,
                                                     keep_supcon=False, orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                                                     W_HOM=W_HOM, W_ORTH=0.0, W_HET_B=W_HET_B, W_CLS=1.0) # W_ORTH=0 here

    # Evaluate classification after each Stage B epoch for monitoring (using eval_epoch_cls defined in Evaluation Functions section)
    val_bce_noorth, val_auc_noorth = eval_epoch_cls(val_loader, model_components=(adapter_noorth, f_hom_noorth, f_id_noorth, clf_noorth))
    print(f"[Stage B NoOrth {ep+1}] total={stats_stageB_noorth['total']:.4f}  val_bce={val_bce_noorth:.4f}  val_auc={val_auc_noorth:.3f}")

# Final evaluation after Stage B without Orthogonality (using run_all_evals defined in Evaluation Functions section)
exp5_eval_results = run_all_evals("Experiment 5: Stage A + B (No Orthogonality)", model_components=(adapter_noorth, f_hom_noorth, f_id_noorth, clf_noorth))
results['Experiment 5 (Stage A + B No Orthogonality)'] = exp5_eval_results


# Experiment 6: Stage A without Adapter

print("\n--- Experiment 6: Stage A without Adapter ---")

# Re-initialize models for a clean run, but skip the adapter
f_hom_noadapter_A   = MLPHead(d, 128).to(DEVICE) # Directly take raw embedding as input
f_id_noadapter_A    = MLPHead(d, 128).to(DEVICE)
# No adapter_noadapter_A
clf_noadapter_A     = FrameClassifier(128).to(DEVICE) # Include clf for run_all_evals structure

# Need optimizers for this specific run
opt_stageA_noadapter = torch.optim.AdamW(
    list(f_hom_noadapter_A.parameters()) + list(f_id_noadapter_A.parameters()), # Only optimize f_hom and f_id
    lr=1e-4, weight_decay=1e-4
)

# Run Stage A without Adapter
print("\nTraining Stage A without Adapter...")
for ep in range(3):
    stats_stageA_noadapter = train_stageA_noadapter(real_id_loader, f_hom_noadapter_A, f_id_noadapter_A, opt_stageA_noadapter,
                                                    orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                                                    W_HOM=W_HOM, W_ORTH=W_ORTH, W_HET_A=W_HET_A)
    print(f"[Stage A NoAdapter {ep+1}] " +
          f"hom={stats_stageA_noadapter['hom']:.4f} orth={stats_stageA_noadapter['orth']:.4f} het={stats_stageA_noadapter['het']:.4f} total={stats_stageA_noadapter['total']:.4f}")

# Evaluate after Stage A without Adapter
# Need a modified run_all_evals or separate calls as adapter is None
# Let's evaluate manually for clarity in this case
print("\n=== EVALUATION: Experiment 6: Stage A without Adapter ===")

mu_noadapter_A, inv_cov_noadapter_A = fit_real_gaussian_noadapter(train_loader, f_hom_noadapter_A)
val_auc_maha_noadapter_A  = eval_distance_auc_noadapter(val_loader, f_hom_noadapter_A, mu_noadapter_A, inv_cov_noadapter_A)
test_auc_maha_noadapter_A = eval_distance_auc_noadapter(test_loader, f_hom_noadapter_A, mu_noadapter_A, inv_cov_noadapter_A)
print(f"[Eval Mahalanobis f_hom (No Adapter Stage A)] val_auc={val_auc_maha_noadapter_A:.3f}  test_auc={test_auc_maha_noadapter_A:.3f}")
results['Experiment 6 (Stage A No Adapter)'] = {'val_auc_maha': val_auc_maha_noadapter_A, 'test_auc_maha': test_auc_maha_noadapter_A}


acc1_zhet_noadapter_A = knn_id_acc_noadapter(train_loader, test_loader, f_id_noadapter_A, embedding_key='z_het', k=1)
acc5_zhet_noadapter_A = knn_id_acc_noadapter(train_loader, test_loader, f_id_noadapter_A, embedding_key='z_het', k=5)
print(f"[Eval KNN f_id (No Adapter Stage A)] test@1={acc1_zhet_noadapter_A:.3f}  test@5={acc5_zhet_noadapter_A:.3f}")
results['Experiment 6 (Stage A No Adapter)'].update({'test_knn_1': acc1_zhet_noadapter_A, 'test_knn_5': acc5_zhet_noadapter_A})


# Experiment 7: Stage A and B with Adapter Ablation

print("\n--- Experiment 7: Stage A + B without Adapter ---")

# Re-initialize models for a clean run, no adapter
f_hom_noadapter_AB   = MLPHead(d, 128).to(DEVICE) # Directly take raw embedding as input
f_id_noadapter_AB    = MLPHead(d, 128).to(DEVICE)
clf_noadapter_AB     = FrameClassifier(128).to(DEVICE)

# Need optimizers for this specific run
opt_stageA_noadapter_AB = torch.optim.AdamW(
    list(f_hom_noadapter_AB.parameters()) + list(f_id_noadapter_AB.parameters()),
    lr=1e-4, weight_decay=1e-4
)
opt_stageB_noadapter_AB = torch.optim.AdamW(
    list(f_hom_noadapter_AB.parameters()) + list(f_id_noadapter_AB.parameters()) + list(clf_noadapter_AB.parameters()),
    lr=1e-4, weight_decay=1e-4
)

# Reuse train_stageA_noadapter for Stage A
print("\nTraining Stage A without Adapter (for Stage A+B)...")
for ep in range(3):
    stats_stageA_noadapter_AB = train_stageA_noadapter(real_id_loader, f_hom_noadapter_AB, f_id_noadapter_AB, opt_stageA_noadapter_AB,
                                                       orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                                                       W_HOM=W_HOM, W_ORTH=W_ORTH, W_HET_A=W_HET_A)
    print(f"[Stage A NoAdapter AB {ep+1}] " +
          f"hom={stats_stageA_noadapter_AB['hom']:.4f} orth={stats_stageA_noadapter_AB['orth']:.4f} het={stats_stageA_noadapter_AB['het']:.4f} total={stats_stageA_noadapter_AB['total']:.4f}")

# Run Stage B without Adapter
print("\nTraining Stage B without Adapter...")
for ep in range(8):
    stats_stageB_noadapter_AB = train_stageB_noadapter(train_loader, f_hom_noadapter_AB, f_id_noadapter_AB, clf_noadapter_AB, opt_stageB_noadapter_AB,
                                                        keep_supcon=False, orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200,
                                                        W_HOM=W_HOM, W_ORTH=W_ORTH, W_HET_B=W_HET_B, W_CLS=1.0)
    val_bce_noadapter_AB, val_auc_noadapter_AB = eval_epoch_cls(val_loader, model_components=(None, f_hom_noadapter_AB, f_id_noadapter_AB, clf_noadapter_AB)) # Pass None for adapter
    print(f"[Stage B NoAdapter AB {ep+1}] total={stats_stageB_noadapter_AB['total']:.4f}  val_bce={val_bce_noadapter_AB:.4f}  val_auc={val_auc_noadapter_AB:.3f}")

# Final evaluation after Stage B without Adapter
# Need a modified run_all_evals or separate calls as adapter is None
print("\n=== EVALUATION: Experiment 7: Stage A + B without Adapter ===")
eval_results_noadapter_AB = {}

# Mahalanobis on f_hom output (trained without adapter)
mu_noadapter_AB, inv_cov_noadapter_AB = fit_real_gaussian_noadapter(train_loader, f_hom_noadapter_AB) # Reuse noadapter fit
val_auc_maha_noadapter_AB  = eval_distance_auc_noadapter(val_loader, f_hom_noadapter_AB, mu_noadapter_AB, inv_cov_noadapter_AB) # Reuse noadapter eval
test_auc_maha_noadapter_AB = eval_distance_auc_noadapter(test_loader, f_hom_noadapter_AB, mu_noadapter_AB, inv_cov_noadapter_AB)
print(f"[Eval Mahalanobis f_hom (No Adapter Stage A+B)] val_auc={val_auc_maha_noadapter_AB:.3f}  test_auc={test_auc_maha_noadapter_AB:.3f}")
eval_results_noadapter_AB['val_auc_maha'] = val_auc_maha_noadapter_AB
eval_results_noadapter_AB['test_auc_maha'] = test_auc_maha_noadapter_AB


val_bce_noadapter_AB, val_auc_cls_noadapter_AB = eval_classification_auc_noadapter(val_loader, f_hom_noadapter_AB, clf_noadapter_AB)
test_bce_noadapter_AB, test_auc_cls_noadapter_AB = eval_classification_auc_noadapter(test_loader, f_hom_noadapter_AB, clf_noadapter_AB)
print(f"[Eval Classification f_hom->clf (No Adapter Stage A+B)] val_auc={val_auc_cls_noadapter_AB:.3f}  test_auc={test_auc_cls_noadapter_AB:.3f}")
print(f"[Eval Classification f_hom->clf (No Adapter Stage A+B)] test_bce={test_bce_noadapter_AB:.4f}  test_auc={test_auc_cls_noadapter_AB:.3f}")
eval_results_noadapter_AB['val_auc_cls'] = val_auc_cls_noadapter_AB
eval_results_noadapter_AB['test_auc_cls'] = test_auc_cls_noadapter_AB
eval_results_noadapter_AB['test_bce_cls'] = test_bce_noadapter_AB


# KNN identity on f_id output (trained without adapter)
# Reuse knn_id_acc_noadapter for this evaluation
acc1_zhet_noadapter_AB = knn_id_acc_noadapter(train_loader, test_loader, f_id_noadapter_AB, embedding_key='z_het', k=1)
acc5_zhet_noadapter_AB = knn_id_acc_noadapter(train_loader, test_loader, f_id_noadapter_AB, embedding_key='z_het', k=5)
print(f"[Eval KNN f_id (No Adapter Stage A+B)] test@1={acc1_zhet_noadapter_AB:.3f}  test@5={acc5_zhet_noadapter_AB:.3f}")
eval_results_noadapter_AB['test_knn_1'] = acc1_zhet_noadapter_AB
eval_results_noadapter_AB['test_knn_5'] = acc5_zhet_noadapter_AB

results['Experiment 7 (Stage A + B No Adapter)'] = eval_results_noadapter_AB


# --- Summarize Results ---
print("\n\n=== Ablation Study Results Summary ===")
for exp_name, exp_results in results.items():
    print(f"\n--- {exp_name} ---")
    for metric, value in exp_results.items():
        print(f"{metric}: {value:.4f}")

--- Running Ablation Studies ---

--- Experiment 1: Raw Embedding Analysis ---

Training Linear Probe...


[Linear Probe Epoch 1]:   0%|          | 0/304 [00:00<?, ?it/s]

[Linear Probe Epoch 1] Avg Loss: 0.3976


[Linear Probe Epoch 2]:   0%|          | 0/304 [00:00<?, ?it/s]

[Linear Probe Epoch 2] Avg Loss: 0.2649


...


--- Evaluating KNN Identity Accuracy on z_het embeddings (No Adapter, k=5) ---


[Collecting z_het embeddings NoAdapter]:   0%|          | 0/304 [00:00<?, ?it/s]

[Collecting z_het embeddings NoAdapter]:   0%|          | 0/36 [00:00<?, ?it/s]

[Eval KNN f_id (No Adapter Stage A+B)] test@1=0.080  test@5=0.053


=== Ablation Study Results Summary ===

--- Experiment 1a (Linear Probe on Raw E) ---
val_auc_cls: 0.9752
test_auc_cls: 0.9724

--- Experiment 1b (KNN on Raw E) ---
test_knn_1: 0.1142
test_knn_5: 0.0664

--- Experiment 1c (Mahalanobis on Raw E) ---
val_auc_maha: 0.8530
test_auc_maha: 0.8577

--- Experiment 2 (Direct Classifier) ---
val_auc_cls: 0.9920
test_auc_cls: 0.9909

--- Experiment 3 (Stage A Only) ---


AttributeError: 'NoneType' object has no attribute 'items'

In [ ]:
# --- Evaluation Functions ---
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm # Ensure tqdm is imported
import numpy as np # Ensure numpy is imported
import torch.nn.functional as F # Ensure F is imported
import torch # Ensure torch is imported
import torch.nn as nn # Ensure nn is imported


### Testing f_hom
### Fit on training REALS only as calibartion test
@torch.no_grad()
def fit_real_gaussian(loader):
    adapter.eval(); f_hom.eval()
    reals = []
    for b in loader:
        e = b["e"].to(DEVICE)
        is_real = b["is_real"].to(DEVICE).bool()
        # Ensure adapter and f_hom are used as in the original model
        z = f_hom(adapter(e))[is_real]
        if z.numel(): reals.append(z.cpu())
    if not reals:
        print("\nWarning: No real samples found in loader for fitting Gaussian.")
        # Return dummy values or raise error if appropriate for downstream use
        # Assuming 'd' is the embedding dimension
        dummy_mu = torch.zeros(f_hom(adapter(torch.zeros(1, d).to(DEVICE))).shape[-1])
        dummy_inv_cov = torch.eye(dummy_mu.shape[-1])
        return dummy_mu, dummy_inv_cov


    Z = torch.cat(reals, dim=0)
    mu = Z.mean(0)
    X  = Z - mu
    C  = (X.T @ X) / max(1, len(Z)-1)
    # Add small diagonal for numerical stability, size should match C
    C  = C + 1e-3 * torch.eye(C.size(0), device=C.device)
    inv_cov = torch.linalg.inv(C)
    return mu, inv_cov

@torch.no_grad()
def mahalanobis(z, mu, inv_cov):
    diff = z - mu.to(z.device)
    # Use matmul instead of einsum for clarity and potential performance
    # Ensure inv_cov is on the correct device
    m2 = torch.sum((diff @ inv_cov.to(z.device)) * diff, dim=1)
    return torch.sqrt(torch.clamp(m2, min=0))

@torch.no_grad()
def eval_distance_auc(loader, mu, inv_cov):
    adapter.eval(); f_hom.eval()
    ys, ds = [], []
    pbar = tqdm(loader, desc="[Eval Mahalanobis]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        # Ensure adapter and f_hom are used as in the original model
        z = f_hom(adapter(e))
        d = mahalanobis(z, mu, inv_cov).cpu().numpy()
        ys.append(y); ds.append(d)
    ys = np.concatenate(ys); ds = np.concatenate(ds)

    # For Mahalanobis distance, lower distance means more likely "real" (label 1).
    # roc_auc_score expects higher scores for the positive class.
    # To use raw distance, flip the labels: 0 becomes 1 (fake), 1 becomes 0 (real).
    if len(np.unique(ys)) < 2:
         print("\nWarning: Only one class present in evaluation labels for Mahalanobis AUC.")
         return float('nan')

    return roc_auc_score(1 - ys, ds) # Flip true labels here

@torch.no_grad()
def embed_features(loader, embedding_key='z_het', model_components=None):
    """
    Collects embeddings ('e' or 'z_het') and associated metadata.
    model_components: tuple (adapter, f_hom, f_id) to allow evaluation of specific model states.
    """
    # Use provided model components or globals if none provided
    adapter_eval = model_components[0] if model_components else adapter
    f_hom_eval = model_components[1] if model_components else f_hom
    f_id_eval = model_components[2] if model_components else f_id

    # Conditionally set to eval mode
    if adapter_eval: adapter_eval.eval()
    if f_hom_eval: f_hom_eval.eval()
    if f_id_eval: f_id_eval.eval()


    Embeddings, ID, IS_REAL = [], [], []
    pbar = tqdm(loader, desc=f"[Collecting {embedding_key} embeddings]", leave=False)
    for b in pbar:
        e = batch["e"].to(DEVICE)
        if embedding_key == 'z_het':
            if adapter_eval and f_id_eval:
                 z = F.normalize(f_id_eval(adapter_eval(e)), dim=-1)
                 Embeddings.append(z.cpu())
            elif f_id_eval: # Case without adapter
                 z = F.normalize(f_id_eval(e), dim=-1)
                 Embeddings.append(z.cpu())
            else:
                 print(f"Warning: Cannot collect {embedding_key} embeddings, f_id_eval not available.")
                 continue

        elif embedding_key == 'e':
            Embeddings.append(e.cpu()) # Use raw embedding 'e'
        elif embedding_key == 'z_hom':
             if adapter_eval and f_hom_eval:
                 z = f_hom_eval(adapter_eval(e))
                 Embeddings.append(z.cpu())
             elif f_hom_eval: # Case without adapter
                 z = f_hom_eval(e)
                 Embeddings.append(z.cpu())
             else:
                 print(f"Warning: Cannot collect {embedding_key} embeddings, f_hom_eval not available.")
                 continue
        else:
            raise ValueError(f"Unknown embedding_key: {embedding_key}")

        ID.append(b["id_idx"].cpu())
        IS_REAL.append(b["is_real"].cpu())

    if not Embeddings: # Handle case where no embeddings were collected
        return torch.empty(0), torch.empty(0), torch.empty(0)

    return torch.cat(Embeddings), torch.cat(ID), torch.cat(IS_REAL)


@torch.no_grad()
def knn_id_acc(train_loader, test_loader, embedding_key='z_het', k=1, model_components=None):
    """
    KNN identity accuracy using specified embeddings.
    gallery = train reals; queries = test reals
    model_components: tuple (adapter, f_hom, f_id) to allow evaluation of specific model states.
    """
    print(f"\n--- Evaluating KNN Identity Accuracy on {embedding_key} embeddings (k={k}) ---")
    # Collect embeddings for gallery (train reals) and queries (test reals)
    Z_tr, ID_tr, R_tr = embed_features(train_loader, embedding_key=embedding_key, model_components=model_components)
    Z_te, ID_te, R_te = embed_features(test_loader, embedding_key=embedding_key, model_components=model_components)


    gallery_Z  = Z_tr[R_tr.bool()]
    gallery_ID = ID_tr[R_tr.bool()]
    query_Z    = Z_te[R_te.bool()]
    query_ID   = ID_te[R_te.bool()]

    if len(gallery_Z) == 0 or len(query_Z) == 0:
        print(f"Warning: Not enough real samples in train or test for KNN evaluation on {embedding_key}.")
        return float('nan')

    # Compute cosine similarity between query embeddings and gallery embeddings
    # Normalize embeddings first for cosine similarity
    gallery_Z_norm = F.normalize(gallery_Z.float(), dim=1)
    query_Z_norm = F.normalize(query_Z.float(), dim=1)

    sims = query_Z_norm @ gallery_Z_norm.T

    # Find k nearest neighbors and predict identity
    if k == 1:
        pred = gallery_ID[sims.argmax(dim=1)]
    else:
        topk = sims.topk(k=k, dim=1).indices
        pred = torch.mode(gallery_ID[topk], dim=1).values

    # Calculate accuracy
    acc = (pred == query_ID).float().mean().item()
    return acc


# Evaluation function for classification AUC and BCE loss
@torch.no_grad()
def eval_epoch_cls(loader, model_components=None):
    """Evaluates classification AUC and BCE loss."""
    # Use provided model components or globals if none provided
    adapter_eval = model_components[0] if model_components else adapter
    f_hom_eval = model_components[1] if model_components else f_hom
    clf_eval = model_components[3] if model_components and len(model_components) > 3 else clf

    # Conditionally set to eval mode
    if adapter_eval: adapter_eval.eval()
    if f_hom_eval: f_hom_eval.eval()
    if clf_eval: clf_eval.eval()


    total_loss = 0.0
    n = 0
    all_logits, all_y = [], []
    bce_loss_fn = nn.BCEWithLogitsLoss() # Corrected typo: BCEWithItsLoss -> BCEWithLogitsLoss


    pbar = tqdm(loader, desc="[Eval CLS]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE); y = batch["y"].to(DEVICE)

        # Handle case where adapter or f_hom might be None in model_components
        if adapter_eval and f_hom_eval:
            z_hom = f_hom_eval(adapter_eval(e))
        elif f_hom_eval: # Case without adapter
            z_hom = f_hom_eval(e)
        else:
            print("Warning: Cannot evaluate classification, f_hom_eval not available.")
            # Return dummy values or skip evaluation if needed
            return float('nan'), float('nan') # Or handle as an error


        if clf_eval:
            logit = clf_eval(z_hom)
            loss = bce_loss_fn(logit, y)
            total_loss += loss.item()*e.size(0); n += e.size(0)
            all_logits.append(logit.sigmoid().cpu())
            all_y.append(y.cpu())
            pbar.set_postfix({"bce": f"{total_loss/max(1,n):.4f}"})
        else:
             print("Warning: Cannot evaluate classification, clf_eval not available.")
             return float('nan'), float('nan')


    import numpy as np
    from sklearn.metrics import roc_auc_score
    y_true = torch.cat(all_y).numpy()
    y_hat  = torch.cat(all_logits).numpy()
    auc = roc_auc_score(y_true, y_hat) if len(np.unique(y_true))>1 else float('nan')
    return total_loss/max(1,n), auc

# Evaluation function for the DirectClassifier (similar to eval_epoch_cls but for DirectClassifier)
@torch.no_grad()
def eval_direct_classifier_auc(loader, model):
    model.eval()
    ys, logits = [], []
    pbar = tqdm(loader, desc="[Eval Direct CLS]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        logit = model(e).cpu().numpy()
        ys.append(y); logits.append(logit)
    ys = np.concatenate(ys); logits = np.concatenate(logits)

    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for Direct Classifier AUC.")
        return float('nan')

    probs = 1 / (1 + np.exp(-logits)) # sigmoid manually
    auc = roc_auc_score(ys, probs)
    return auc

# Evaluation function for the Linear Probe
@torch.no_grad()
def eval_linear_probe_auc(loader, model):
    """Evaluates classification AUC for the linear probe."""
    model.eval()
    ys, logits = [], []
    pbar = tqdm(loader, desc="[Eval Linear Probe]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        logit = model(e).cpu().numpy()
        ys.append(y); logits.append(logit)
    ys = np.concatenate(ys); logits = np.concatenate(logits)

    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for Linear Probe AUC.")
        return float('nan')

    probs = 1 / (1 + np.exp(-logits)) # sigmoid manually
    auc = roc_auc_score(ys, probs)
    return auc

# Evaluation function for the linear probe on z_hom
@torch.no_grad()
def eval_linear_probe_zhom_auc(loader, adapter_m, f_hom_m, linear_probe_m):
    # Conditionally set to eval mode
    if adapter_m: adapter_m.eval()
    if f_hom_m: f_hom_m.eval()
    linear_probe_m.eval()

    ys, logits = [], []
    pbar = tqdm(loader, desc="[Eval LP on z_hom]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        with torch.no_grad():
            # Handle case where adapter might be None
            if adapter_m and f_hom_m:
                z_hom = f_hom_m(adapter_m(e))
            elif f_hom_m: # Case without adapter
                 z_hom = f_hom_m(e)
            else:
                print("Warning: Cannot evaluate LP on z_hom, f_hom_m not available.")
                continue # Skip this batch

        logit = linear_probe_m(z_hom).cpu().numpy()
        ys.append(y); logits.append(logit)
    ys = np.concatenate(ys); logits = np.concatenate(logits)

    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for LP on z_hom AUC.")
        return float('nan')

    probs = 1 / (1 + np.exp(-logits)) # sigmoid manually
    auc = roc_auc_score(ys, probs)
    return auc

# Evaluation function for fitting Gaussian on raw real embeddings
@torch.no_grad()
def fit_raw_real_gaussian(loader):
    """Fits a Gaussian on raw real embeddings from the training set."""
    reals = []
    pbar = tqdm(loader, desc="[Fit Raw Real Gaussian]", leave=False)
    for b in pbar:
        e = b["e"].to(DEVICE)
        is_real = b["is_real"].to(DEVICE).bool()
        raw_reals = e[is_real]
        if raw_reals.numel(): reals.append(raw_reals.cpu())

    if not reals:
        print("\nWarning: No real samples found in loader for fitting Raw Gaussian.")
        # Assuming 'd' is the embedding dimension
        dummy_mu = torch.zeros(d)
        dummy_inv_cov = torch.eye(d)
        return dummy_mu, dummy_inv_cov

    Z = torch.cat(reals, dim=0)
    mu = Z.mean(0)
    X  = Z - mu
    C  = (X.T @ X) / max(1, len(Z)-1)
    C  = C + 1e-3 * torch.eye(C.size(0), device=C.device)
    inv_cov = torch.linalg.inv(C)
    return mu, inv_cov

# Evaluation function for Mahalanobis AUC on raw embeddings
@torch.no_grad()
def eval_raw_distance_auc(loader, mu, inv_cov):
    """Evaluates Mahalanobis AUC on raw embeddings."""
    ys, ds = [], []
    pbar = tqdm(loader, desc="[Eval Raw Mahalanobis]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE)
        y = batch["y"].cpu().numpy()
        d = mahalanobis(e, mu, inv_cov).cpu().numpy() # Use raw embedding 'e'
        ys.append(y); ds.append(d)
    ys = np.concatenate(ys); ds = np.concatenate(ds)

    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for Raw Mahalanobis AUC.")
        return float('nan')

    # Flip labels as higher distance means more likely fake (label 0)
    return roc_auc_score(1 - ys, ds)

# Evaluation function for embedding features without adapter
@torch.no_grad()
def embed_features_noadapter(loader, f_hom_m=None, f_id_m=None, embedding_key='z_het'):
    """Collects embeddings ('e', f_hom(e) or f_id(e)) and associated metadata, no adapter."""
    if f_hom_m: f_hom_m.eval()
    if f_id_m: f_id_m.eval()

    Embeddings, ID, IS_REAL = [], [], []
    pbar = tqdm(loader, desc=f"[Collecting {embedding_key} embeddings NoAdapter]", leave=False)
    for b in pbar:
        e = b["e"].to(DEVICE) # Use raw embedding directly

        if embedding_key == 'z_het' and f_id_m:
            z = F.normalize(f_id_m(e), dim=-1) # Use f_id directly on raw embedding
            Embeddings.append(z.cpu())
        elif embedding_key == 'z_hom' and f_hom_m:
             z = f_hom_m(e) # Use f_hom directly on raw embedding
             Embeddings.append(z.cpu())
        elif embedding_key == 'e':
            Embeddings.append(e.cpu()) # Use raw embedding 'e'
        else:
            print(f"Warning: Cannot collect {embedding_key} embeddings, required model not available.")
            continue # Skip this batch

        ID.append(b["id_idx"].cpu())
        IS_REAL.append(b["is_real"].cpu())

    if not Embeddings: # Handle case where no embeddings were collected
        return torch.empty(0), torch.empty(0), torch.empty(0)


    return torch.cat(Embeddings), torch.cat(ID), torch.cat(IS_REAL)

# Evaluation function for KNN identity accuracy without adapter
@torch.no_grad()
def knn_id_acc_noadapter(train_loader, test_loader, f_id_m, embedding_key='z_het', k=1):
    """
    KNN identity accuracy using specified embeddings (no adapter).
    gallery = train reals; queries = test reals
    """
    print(f"\n--- Evaluating KNN Identity Accuracy on {embedding_key} embeddings (No Adapter, k={k}) ---")
    # Collect embeddings for gallery (train reals) and queries (test reals)
    Z_tr, ID_tr, R_tr = embed_features_noadapter(train_loader, f_id_m=f_id_m, embedding_key=embedding_key)
    Z_te, ID_te, R_te = embed_features_noadapter(test_loader, f_id_m=f_id_m, embedding_key=embedding_key)


    gallery_Z  = Z_tr[R_tr.bool()]
    gallery_ID = ID_tr[R_tr.bool()]
    query_Z    = Z_te[R_te.bool()]
    query_ID   = ID_te[R_te.bool()]

    if len(gallery_Z) == 0 or len(query_Z) == 0:
        print(f"Warning: Not enough real samples in train or test for KNN evaluation on {embedding_key} (No Adapter).")
        return float('nan')

    # Compute cosine similarity between query embeddings and gallery embeddings
    gallery_Z_norm = F.normalize(gallery_Z.float(), dim=1)
    query_Z_norm = F.normalize(query_Z.float(), dim=1)

    sims = query_Z_norm @ gallery_Z_norm.T

    # Find k nearest neighbors and predict identity
    if k == 1:
        pred = gallery_ID[sims.argmax(dim=1)]
    else:
        topk = sims.topk(k=k, dim=1).indices
        pred = torch.mode(gallery_ID[topk], dim=1).values

    # Calculate accuracy
    acc = (pred == query_ID).float().mean().item()
    return acc

# Evaluation function for classification AUC and BCE loss without adapter
@torch.no_grad()
def eval_classification_auc_noadapter(loader, f_hom_m, clf_m):
    f_hom_m.eval(); clf_m.eval()
    ys, logits = [], []
    bce_loss_fn = nn.BCEWithLogitsLoss() # Corrected typo: BCEWithItsLoss -> BCEWithLogitsLoss
    total_loss = 0.0
    n = 0
    pbar = tqdm(loader, desc="[Eval CLS AUC NoAdapter]", leave=False)
    for batch in pbar:
        e = batch["e"].to(DEVICE) # Use raw embedding
        y = batch["y"].to(DEVICE)
        z_hom = f_hom_m(e) # Use f_hom directly on raw embedding
        logit = clf_m(z_hom)
        loss = bce_loss_fn(logit, y)
        total_loss += loss.item()*e.size(0); n += e.size(0)
        ys.append(y.cpu().numpy()); logits.append(logit.sigmoid().cpu().numpy())
    ys = np.concatenate(ys); logits = np.concatenate(logits)
    if len(np.unique(ys)) < 2:
        print("\nWarning: Only one class present in evaluation labels for Classification AUC (No Adapter).")
        return total_loss/max(1,n), float('nan')
    auc = roc_auc_score(ys, logits)
    return total_loss/max(1,n), auc


def run_all_evals(stage_name="Current Stage", model_components=None):
    """
    Runs a standard set of evaluations and returns a dictionary of results.
    model_components: tuple (adapter, f_hom, f_id, clf) to allow evaluation of specific model states.
    """
    print(f"\n=== EVALUATION: {stage_name} ===")
    eval_results = {}

    # Mahalanobis (fit on TRAIN reals) - uses z_hom
    # Need to use model components consistently if provided
    # Check if adapter is present in model_components (assuming adapter is the first element)
    adapter_present = model_components and len(model_components) > 0 and model_components[0] is not None

    if adapter_present:
        mu, inv_cov = fit_real_gaussian(train_loader) # fit_real_gaussian uses adapter and f_hom
        val_auc_maha  = eval_distance_auc(val_loader,  mu, inv_cov) # eval_distance_auc uses adapter and f_hom
        test_auc_maha = eval_distance_auc(test_loader, mu, inv_cov)
        print(f"[Eval Mahalanobis z_hom]  val_auc={val_auc_maha:.3f}  test_auc={test_auc_maha:.3f}")
        eval_results['val_auc_maha'] = val_auc_maha
        eval_results['test_auc_maha'] = test_auc_maha
    else:
        # If no adapter, Mahalanobis is evaluated on f_hom output directly
        # Need to use the no-adapter evaluation functions
        f_hom_eval = model_components[1] if model_components and len(model_components) > 1 else f_hom # Get f_hom from components or global
        if f_hom_eval:
            mu_noadapter, inv_cov_noadapter = fit_real_gaussian_noadapter(train_loader, f_hom_eval)
            val_auc_maha_noadapter  = eval_distance_auc_noadapter(val_loader, f_hom_eval, mu_noadapter, inv_cov_noadapter)
            test_auc_maha_noadapter = eval_distance_auc_noadapter(test_loader, f_hom_eval, mu_noadapter, inv_cov_noadapter)
            print(f"[Eval Mahalanobis f_hom (No Adapter)] val_auc={val_auc_maha_noadapter:.3f}  test_auc={test_auc_maha_noadapter:.3f}")
            eval_results['val_auc_maha'] = val_auc_maha_noadapter
            eval_results['test_auc_maha'] = test_auc_maha_noadapter
        else:
             print("Skipping Mahalanobis AUC: f_hom not available.")
             eval_results['val_auc_maha'] = float('nan')
             eval_results['test_auc_maha'] = float('nan')


    # Classification AUC (uses z_hom and clf)
    # Pass model components to eval_epoch_cls if they are provided
    print("\n--- Evaluating Classification AUC ---")
    # Check if clf is available in model_components or globals and trainable
    clf_present = (model_components and len(model_components) > 3 and model_components[3] is not None) or ('clf' in globals() and any(p.requires_grad for p in clf.parameters()))

    if clf_present:
         if adapter_present:
             val_bce, val_auc_cls = eval_epoch_cls(val_loader, model_components=model_components)
             test_bce, test_auc_cls = eval_epoch_cls(test_loader, model_components=model_components) # Added test evaluation
         else:
             # If no adapter, evaluate classification using the no-adapter function
             f_hom_eval = model_components[1] if model_components and len(model_components) > 1 else f_hom
             clf_eval = model_components[3] if model_components and len(model_components) > 3 else clf
             if f_hom_eval and clf_eval:
                 val_bce, val_auc_cls = eval_classification_auc_noadapter(val_loader, f_hom_eval, clf_eval)
                 test_bce, test_auc_cls = eval_classification_auc_noadapter(test_loader, f_hom_eval, clf_eval)
             else:
                 print("Skipping Classification AUC: f_hom or clf not available.")
                 val_bce, val_auc_cls = float('nan'), float('nan')
                 test_bce, test_auc_cls = float('nan'), float('nan')


         print(f"[Eval Classification z_hom->clf] val_auc={val_auc_cls:.3f}  test_auc={test_auc_cls:.3f}")
         print(f"[Eval Classification z_hom->clf] test_bce={test_bce:.4f}  test_auc={test_auc_cls:.3f}") # Print test BCE
         eval_results['val_auc_cls'] = val_auc_cls
         eval_results['test_auc_cls'] = test_auc_cls
         eval_results['test_bce_cls'] = test_bce
    else:
        print("Skipping Classification AUC: Classifier not available or not trainable.")
        eval_results['val_auc_cls'] = float('nan')
        eval_results['test_auc_cls'] = float('nan')
        eval_results['test_bce_cls'] = float('nan')


    # KNN identity on z_het (gallery=train reals, queries=test reals)
    # Uses the consolidated knn_id_acc function, pass model components if provided
    if adapter_present:
        acc1_zhet = knn_id_acc(train_loader, test_loader, embedding_key='z_het', k=1, model_components=model_components)
        acc5_zhet = knn_id_acc(train_loader, test_loader, embedding_key='z_het', k=5, model_components=model_components)
    else:
        # If no adapter, evaluate KNN using the no-adapter function on f_id output
        f_id_eval = model_components[2] if model_components and len(model_components) > 2 else f_id # Get f_id from components or global
        if f_id_eval:
            acc1_zhet = knn_id_acc_noadapter(train_loader, test_loader, f_id_eval, embedding_key='z_het', k=1)
            acc5_zhet = knn_id_acc_noadapter(train_loader, test_loader, f_id_eval, embedding_key='z_het', k=5)
        else:
            print("Skipping KNN Identity: f_id not available.")
            acc1_zhet = float('nan')
            acc5_zhet = float('nan')


    print(f"[Eval KNN z_het]  test@1={acc1_zhet:.3f}  test@5={acc5_zhet:.3f}")
    eval_results['test_knn_1'] = acc1_zhet
    eval_results['test_knn_5'] = acc5_zhet

    return eval_results


# Placeholder for the original run_all_evals call at the end of training cell
# This will be moved to the final combined execution cell later.
# print("\n=== FINAL EVAL AFTER STAGE B ===")
# run_all_evals("Final Model")

In [ ]:
# --- Stage A: warm-up (no classifier) ---
for ep in range(3):
    stats = train_epoch_stageA(train_loader, orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200)
    print(f"[Warmup] epoch {ep+1}: " +
          f"hom={stats['hom']:.4f} orth={stats['orth']:.4f} het={stats['het']:.4f} total={stats['total']:.4f}")

# (Optional) run your Mahalanobis AUC and z_het KNN here
# ✅ Run evals after Stage A
print("\n=== EVAL AFTER STAGE A ===")
run_all_evals()

# --- Stage B: add classifier ---
for ep in range(8):
    stats = train_epoch_stageB(train_loader, keep_supcon=False, orth_on_reals_only=True, max_pos=4, max_neg=32, log_every=200)
    val_bce, val_auc = eval_epoch_cls(val_loader)
    print(f"[CLS] epoch {ep+1}: total={stats['total']:.4f}  val_bce={val_bce:.4f}  val_auc={val_auc:.3f}")

# ✅ Final evals after Stage B
print("\n=== FINAL EVAL AFTER STAGE B ===")
run_all_evals()

In [ ]:
#### Visualizations, other datasets, metrics?

### PCA of z_hom embeddings maybe? Or statistic on the embedding space.